# CMIN protocol run — filling Tables 4, 5, and Sections 6–7

Produces the numbers the protocol document leaves as `TODO` and `–`, so the paper can be
written against real output rather than placeholders.

| protocol section | what this notebook emits |
|---|---|
| Table 4 — baseline families | fitted results for every family, or an explicit skip reason |
| Table 5 — multi-horizon MAE | mean ± sd over seeds at h = 1, 3, 7, 14, 30 |
| 5.3 — ablation ladder | all 9 rungs, sharing one temporal encoder and head |
| 5.5 — metrics | MAE, RMSE, MASE, skill vs persistence, per-market spread, worst decile, high-volatility, spikes |
| 6.1–6.3 — forecasting results | main table, ablation table, robustness |
| 7.1–7.4 — graph validation | seed stability, external alignment, event-aligned change, null controls |

### Cell map

**Cells 1–13 are copied verbatim from `CMIN_v4.ipynb`** — data loading, feature engineering,
the CMIN model, windowing, and the training loop. They keep v4's own section labels
(`=== 1.`, `=== 3b.`, `=== 5c.`, `=== 6b.` …), which are neither sequential nor unique,
because renumbering them would break the correspondence to v4. Nothing there is
re-implemented, so a difference in results cannot come from a difference in the pipeline.

**Cells 14–21 are new** and labelled `P1`–`P8` so their numbering cannot be confused with
v4's:

| cell | label | does |
|---|---|---|
| 14 | `P1` | ablation ladder, graph modes, `SMOKE` switch |
| 15 | `P2` | binds the ladder to the v4 training machinery |
| 16 | `P3` | statistical and ML baselines |
| 17 | `P4` | registers the neural and graph baselines |
| 18 | `P5` | metric suite |
| 19 | `P6` | runs everything, resumable |
| 20 | `P7` | Tables 4 and 5, decisive contrast |
| 21 | `P8` | graph validation |

23 cells in total: 1 intro, 21 code, 1 closing note.

### The rung that matters

Rung 5 is **full numerical context as features, adjacency frozen to `A_phys`**. Compared
against rung 6 (full context-conditioned graph) it isolates the paper's actual claim:

> does context help *because it reshapes the graph*, or merely because it enlarges the
> input?

v4's existing ablation cannot answer this — every variant there removes capacity as well as
graph conditioning, so the two explanations stay confounded. If rung 5 matches rung 6, the
context-conditioning mechanism is not carrying the result and the paper's framing has to
change. Run it early.

### Before the long run

Set `SMOKE = True` (cell **P1**) first. It runs one seed and a handful of epochs across every
model and rung, so import errors, shape errors and missing files surface in minutes rather
than four hours in. Then set it to `False`.

Every model is registered independently and wrapped: one failure records a reason and the
suite continues. Checkpoints are per `(model, seed)` on Drive, so a Colab disconnect costs
only the run in flight.

### Honest scope note for the write-up

The graph baselines here are **compact re-implementations** of STGCN, AGCRN and Graph
WaveNet against this data contract, not the authors' released code. Say so in the paper.
They are matched on temporal encoder width, optimiser, budget and model-selection rule, so
the comparison is fair in the sense that matters, but they are not the reference
implementations and should not be described as such.

In [1]:
# === 1. Imports, device, config, and Colab upload ===
import os, math, time, re, itertools, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU detected - training will be slow. Runtime -> Change runtime type -> GPU.')

CFG = dict(
    L=30, H=30, HORIZONS=[1,3,7,14,30],
    d=64, d_ctx=32,
    lr=1e-3, weight_decay=1e-4, grad_clip=5.0,
    epochs=15, patience=5, batch=32, n_seeds=3,
    sigma_quantile=0.25, knn=8,
    regime_horizon=7,
    context_alpha=True,
    lambda_context_entropy=1e-3,
    lambda_alpha_phys=5e-3,
    edge_threshold=0.02,
    baseline_epochs=8, baseline_patience=3, run_baselines=True,
    bootstrap_B=1000,
    policy_half_life_days=90,
)

# --- scenario-suite knobs -------------------------------------------------------------
SCFG = dict(
    seeds=10,             # every scenario
    epochs=50,
    patience=10,
    spike_threshold=0.10, # |log move| defining a price spike
    news_half_life=7,     # default decay; S2 sweeps this
    news_share_norm=True, # divide national counts by daily indexed volume
    horizon_weight_clip=5.0,
    lambda_spike=0.3,     # weight of the S6 auxiliary BCE term
    bootstrap_B=1000,
    lambda_rest=1.0,      # weight on the REST-TS residual-supervision term
)


# ---------------------------------------------------------------------------------
# Google Drive is mounted HERE (not later) so the input files can be resolved from it.
# Upload the 10 inputs to MyDrive/cmin_data ONCE; every future session — and any other
# Google account — picks them up automatically. Checkpoints and outputs also live on
# Drive, so a disconnect costs one run rather than the suite.
# ---------------------------------------------------------------------------------
import shutil

NEEDED = [
    'dataset_train.xlsx', 'dataset_test.xlsx', 'Matrix_Jarak_Jawa_Timur_simetris.xlsx',
    'nasa_power_jawa_timur_merged.csv', '06_ConsumerPriceIndicies.xlsx',
    'policy_event_features.csv', 'policy_extraction_review_queue.csv',
    'policy_feature_dictionary.csv',
    'news_channels_national.csv.gz', 'news_channels_city.csv.gz',
]

DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
except Exception:
    print('not on Colab (or Drive refused) - using local paths')

DATA_DIR = f'{DRIVE}/cmin_data'   if DRIVE else '.'
CKPT_DIR = f'{DRIVE}/cmin_v4_ckpt' if DRIVE else 'cmin_v4_ckpt'
OUT_DIR  = f'{DRIVE}/cmin_v4_out'  if DRIVE else '.'
for d in ([DATA_DIR, CKPT_DIR, OUT_DIR] if DRIVE else [CKPT_DIR]):
    os.makedirs(d, exist_ok=True)

# 1) pull whatever Drive already has into the working directory
if DRIVE:
    for f in NEEDED:
        src = os.path.join(DATA_DIR, f)
        if not os.path.exists(f) and os.path.exists(src):
            shutil.copy(src, f)

# 2) ask for anything still missing, then persist it to Drive for next time
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'{len(missing)} file(s) not found in {DATA_DIR}:')
    for f in missing: print('   -', f)
    try:
        from google.colab import files
        print('\nUpload them now (they will be saved to Drive automatically):')
        files.upload()
    except Exception:
        raise FileNotFoundError(f'Place these next to the notebook: {missing}')
    if DRIVE:
        for f in NEEDED:
            if os.path.exists(f) and not os.path.exists(os.path.join(DATA_DIR, f)):
                shutil.copy(f, os.path.join(DATA_DIR, f))
                print(f'  saved to Drive: {f}')

missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f'still missing: {missing}'
print(f'\nall {len(NEEDED)} inputs ready')
print(f'  data       {DATA_DIR}')
print(f'  checkpoints{"":1s} {CKPT_DIR}')
print(f'  outputs    {OUT_DIR}')

device: cuda
Mounted at /content/drive

all 10 inputs ready
  data       /content/drive/MyDrive/cmin_data
  checkpoints  /content/drive/MyDrive/cmin_v4_ckpt
  outputs    /content/drive/MyDrive/cmin_v4_out


In [2]:
# === 2. Load price data; join NASA/POWER weather; load Food CPI and policy events ===
tr = pd.read_excel('dataset_train.xlsx')
te = pd.read_excel('dataset_test.xlsx')
for d in (tr, te):
    d['tanggal'] = pd.to_datetime(d['tanggal'])
df = pd.concat([tr, te], ignore_index=True).sort_values('tanggal').copy()

# NASA/POWER is daily and location-specific. Prefixing avoids silently reusing weather
# columns that happened to be present in the original price workbook.
nasa = pd.read_csv('nasa_power_jawa_timur_merged.csv', parse_dates=['date'])
# Every meteorological column the product ships with, kept for reporting in E2.
NASA_META_COLS = ['wilayah', 'date', 'latitude', 'longitude', 'elevation_m', 'source_file', 'YEAR', 'DOY']
NASA_ALL_VARS = [c for c in nasa.columns if c not in NASA_META_COLS]
NASA_DATE_MIN, NASA_DATE_MAX, NASA_ROWS = nasa['date'].min(), nasa['date'].max(), len(nasa)
NASA_FEATURES = ['PRECTOTCORR', 'GWETTOP', 'PS', 'QV2M', 'TS', 'T2M', 'T2M_MIN', 'T2M_MAX', 'RH2M', 'WS2M']
assert set(NASA_FEATURES).issubset(nasa.columns)
nasa = nasa.rename(columns={'wilayah': 'kab_kota', 'date': 'tanggal'})
nasa = nasa[['kab_kota', 'tanggal', *NASA_FEATURES]].copy()
nasa = nasa.rename(columns={c: f'nasa_{c}' for c in NASA_FEATURES})
assert not nasa.duplicated(['kab_kota', 'tanggal']).any(), 'NASA key must be unique'
df = df.merge(nasa, on=['kab_kota', 'tanggal'], how='left', validate='many_to_one')
nasa_cols = [f'nasa_{c}' for c in NASA_FEATURES]
missing_weather = int(df[nasa_cols].isna().any(axis=1).sum())
assert missing_weather == 0, f'{missing_weather} price rows lack NASA weather after the merge'

MONTH_NUM = {m: i for i, m in enumerate([
    'january','february','march','april','may','june',
    'july','august','september','october','november','december'], start=1)}

def load_food_cpi(path):
    """Parse BI's presentation-format CPI workbook and return its Foods component.

    Base-year changes make raw CPI levels incomparable across regimes, so the model uses
    only MoM/YoY percentage changes when both observations have the same CPI base.
    """
    raw = pd.read_excel(path, sheet_name=0, header=None)
    label_row = raw.iloc[5].fillna('').astype(str).str.strip().str.lower()
    food_col = label_row[label_row.eq('foods')].index[0]
    period_col = 3  # 'End of Period' field in the BI sheet
    rows, year, base_id = [], None, None
    for _, row in raw.iterrows():
        label = str(row.iloc[period_col]).strip()
        year_match = re.search(r'\b(19|20)\d{2}\b', label)
        if year_match:
            year = int(year_match.group())
            base_match = re.search(r'(\d{4}\s*=\s*100)', label)
            base_id = base_match.group(1).replace(' ', '') if base_match else label
        month = MONTH_NUM.get(label.lower())
        value = pd.to_numeric(row.iloc[food_col], errors='coerce')
        if year and month and pd.notna(value):
            rows.append((pd.Timestamp(year, month, 1), float(value), base_id))
    cpi = pd.DataFrame(rows, columns=['month', 'food_cpi_level', 'cpi_base']).drop_duplicates('month', keep='last').sort_values('month')
    cpi['food_cpi_mom_pct'] = 100 * cpi['food_cpi_level'].pct_change()
    cpi.loc[cpi['cpi_base'].ne(cpi['cpi_base'].shift()), 'food_cpi_mom_pct'] = np.nan
    cpi['food_cpi_yoy_pct'] = 100 * (cpi['food_cpi_level'] / cpi['food_cpi_level'].shift(12) - 1)
    cpi.loc[cpi['cpi_base'].ne(cpi['cpi_base'].shift(12)), 'food_cpi_yoy_pct'] = np.nan
    # The value released for a calendar month is treated as observable from the next
    # month. This is intentionally conservative for daily forecasts.
    cpi['month_available'] = cpi['month'] + pd.offsets.MonthBegin(1)
    return cpi[['month_available', 'food_cpi_mom_pct', 'food_cpi_yoy_pct']]

food_cpi = load_food_cpi('06_ConsumerPriceIndicies.xlsx')
df['cpi_month'] = df['tanggal'].dt.to_period('M').dt.to_timestamp()
df = df.merge(food_cpi, left_on='cpi_month', right_on='month_available', how='left', validate='many_to_one').drop(columns=['month_available'])

# Audit CSVs are loaded for provenance; only policy_event_features.csv supplies model inputs.
policy_dictionary = pd.read_csv('policy_feature_dictionary.csv')
policy_review_queue = pd.read_csv('policy_extraction_review_queue.csv')
policy_events_all = pd.read_csv('policy_event_features.csv', parse_dates=['event_date'])
policy_events = policy_events_all.query('include_in_model == 1').copy()
assert not policy_events['event_date'].isna().any()
POLICY_EVENT_COLS = [
    'expected_price_pressure', 'price_relevance',
    'rice', 'chili', 'shallot', 'garlic', 'beef', 'buffalo_meat', 'chicken_meat',
    'chicken_eggs', 'sugar', 'cooking_oil', 'soybean', 'corn', 'wheat', 'fish',
    'price_management', 'supply_reserve', 'procurement_distribution', 'import_export',
    'market_operation', 'production', 'subsidy_assistance',
]
assert set(POLICY_EVENT_COLS).issubset(policy_events.columns)

def policy_decay_features(event_table, daily_dates, cols, half_life_days=90):
    """National policy features; events begin on the following day to avoid look-ahead."""
    idx = pd.DatetimeIndex(daily_dates)
    ev = event_table.groupby('event_date')[cols].sum().reindex(idx, fill_value=0.0)
    ev = ev.shift(1, fill_value=0.0)  # no same-day policy information in the forecast input
    decay = 0.5 ** (1.0 / half_life_days)
    out = np.zeros((len(idx), len(cols)), dtype=np.float32)
    state = np.zeros(len(cols), dtype=np.float32)
    for t, values in enumerate(ev.to_numpy(np.float32)):
        state = decay * state + values
        out[t] = state
    return pd.DataFrame(out, index=idx, columns=[f'policy_{c}_90d' for c in cols])

daily_policy = policy_decay_features(policy_events, pd.DatetimeIndex(sorted(df['tanggal'].unique())), POLICY_EVENT_COLS, CFG['policy_half_life_days'])

# a node is a (city, market) pair, because market names repeat across cities
df['node'] = df['kab_kota'].astype(str) + ' | ' + df['pasar'].astype(str)
nodes = sorted(df['node'].unique())
cities = sorted(df['kab_kota'].unique())
N, R = len(nodes), len(cities)
dates = pd.DatetimeIndex(np.sort(df['tanggal'].unique()))
T = len(dates)
TRAIN_MAX = pd.Timestamp(tr['tanggal'].max())
TEST_START = pd.Timestamp(te['tanggal'].min())
print(f'N(nodes)={N}  R(cities)={R}  T(days)={T}  train<= {TRAIN_MAX.date()}  test>= {TEST_START.date()}')
print(f'NASA variables available: {len(NASA_ALL_VARS)}; joined: {len(NASA_FEATURES)}; rows: {NASA_ROWS:,} '
      f'({NASA_DATE_MIN.date()} to {NASA_DATE_MAX.date()})')
print(f'Food CPI range: {food_cpi.month_available.min().date()} to {food_cpi.month_available.max().date()}')
print(f'policy documents: {len(policy_events_all)} retrieved, {len(policy_events)} used; '
      f'review-queue records: {len(policy_review_queue)}')

# assignment matrix S (node -> city) and its column-normalised pooling form
cidx = {c:i for i,c in enumerate(cities)}
node_city = df.drop_duplicates('node').set_index('node')['kab_kota'].to_dict()
S = np.zeros((N, R), np.float32)
for j,n in enumerate(nodes):
    S[j, cidx[node_city[n]]] = 1.0
Sbar = S / S.sum(0, keepdims=True)

# physical graph from the distance matrix (local Gaussian kernel + kNN sparsify)
def _norm(s):
    return (str(s).replace('Kabupaten ','').replace('Kota ','')
            .replace(' Kab.','').replace('Kab. ','').strip().lower())
Dm = pd.read_excel('Matrix_Jarak_Jawa_Timur_simetris.xlsx')
dist_cities = list(Dm.iloc[:,0]); M = Dm.iloc[:,1:].to_numpy(float)
sigma = np.quantile(M[M>0], CFG['sigma_quantile'])
A = np.exp(-(M**2)/sigma**2); np.fill_diagonal(A, 0.0)
if CFG['knn']:
    keep = np.zeros_like(A)
    for a in range(R):
        ix = np.argsort(A[a])[-CFG['knn']:]; keep[a, ix] = A[a, ix]
    A = keep
A[A < 1e-3] = 0.0
rs = A.sum(1, keepdims=True); rs[rs==0] = 1.0; A = A / rs
perm = np.array([{_norm(c):i for i,c in enumerate(dist_cities)}[_norm(c)] for c in cities])
A_phys = A[np.ix_(perm, perm)].astype(np.float32)
print('A_phys', A_phys.shape, 'density', round(float((A_phys>0).mean()),3))

N(nodes)=120  R(cities)=38  T(days)=1826  train<= 2025-02-27  test>= 2025-02-28
NASA variables available: 20; joined: 10; rows: 90,934 (2020-01-01 to 2026-07-20)
Food CPI range: 1996-02-01 to 2026-07-01
policy documents: 91 retrieved, 22 used; review-queue records: 3
A_phys (38, 38) density 0.197


In [3]:
# === 3. Leakage-safe feature engineering -> full arrays ===
def panel(col):
    return df.pivot_table(index='tanggal', columns='node', values=col,
                          aggfunc='first').reindex(index=dates, columns=nodes)

def fill_time(P):
    return pd.DataFrame(P).ffill().bfill().to_numpy()

def z_from_array(P):
    P = np.asarray(P, float)
    m = np.nanmean(P[istrain]); s = np.nanstd(P[istrain])
    if not np.isfinite(s) or s < 1e-6: s = 1.0
    return np.nan_to_num(fill_time((P - m) / s)).astype(np.float32)

def zfeat(col):
    return z_from_array(panel(col).to_numpy(float))

def anomaly_feat(col):
    P = panel(col)
    train = P.iloc[np.where(istrain)[0]]
    clim = train.groupby(train.index.dayofyear).mean()
    clim = clim.reindex(np.arange(1, 367)).interpolate(limit_direction='both').ffill().bfill()
    baseline = clim.reindex(dates.dayofyear).to_numpy(float)
    return z_from_array(P.to_numpy(float) - baseline)

price = panel('harga').to_numpy(float)
istrain = np.asarray(dates <= TRAIN_MAX)

def winsor(x, lo=1, hi=99):
    out = x.copy()
    for j in range(x.shape[1]):
        col = x[istrain, j]; col = col[~np.isnan(col)]
        if len(col) > 10:
            a, b = np.percentile(col, [lo, hi]); out[:, j] = np.clip(out[:, j], a, b)
    return out

Lp = np.log(winsor(price)); obs = (~np.isnan(price)).astype(np.float32)
mu = np.nanmean(np.where(istrain[:,None], Lp, np.nan), 0)
sd = np.nanstd(np.where(istrain[:,None], Lp, np.nan), 0); sd[sd < 1e-6] = 1.0
Z = (Lp - mu) / sd
Zf = fill_time(Z).astype(np.float32); Y = np.nan_to_num(Z).astype(np.float32)

# NASA/POWER weather is the authoritative weather source for this augmented model.
weather_cols = ['nasa_PRECTOTCORR','nasa_GWETTOP','nasa_PS','nasa_QV2M','nasa_TS','nasa_T2M','nasa_RH2M','nasa_WS2M']
weather = [zfeat(c) for c in weather_cols]
weather_anom = [anomaly_feat(c) for c in weather_cols]
rain = weather[0]; rain_anom = weather_anom[0]
harvest_raw = panel('Panen tahun lalu').to_numpy(float)
harvest = z_from_array(harvest_raw)
harvest_pressure = np.maximum(harvest, 0).astype(np.float32)
rain_harvest = z_from_array(rain_anom * harvest_pressure)
harvest_hi = np.nanquantile(harvest_raw[istrain], 0.75)
production_season = np.nan_to_num((harvest_raw >= harvest_hi).astype(np.float32))
agro = [harvest, harvest_pressure, rain_harvest, production_season]

rain_city_mean = np.nanmean(panel('nasa_PRECTOTCORR').to_numpy(float), axis=1)
harvest_city_mean = np.nanmean(harvest_raw, axis=1)
peak_rain_flag = (rain_city_mean >= np.nanquantile(rain_city_mean[istrain], 0.75)).astype(int)
harvest_flag = (harvest_city_mean >= np.nanquantile(harvest_city_mean[istrain], 0.75)).astype(int)

city_harvest_train, city_market_count = [], []
for c in cities:
    ix = np.where(S[:, cidx[c]] > 0)[0]
    city_harvest_train.append(np.nanmean(harvest_raw[istrain][:, ix])); city_market_count.append(len(ix))
city_harvest_train = np.asarray(city_harvest_train, float); city_market_count = np.asarray(city_market_count, float)
if np.isnan(city_harvest_train).any(): city_harvest_train = np.nan_to_num(city_harvest_train, nan=np.nanmedian(city_harvest_train))
production_city_mask = city_harvest_train >= np.nanquantile(city_harvest_train, 0.75)
prod_s = np.nanstd(city_harvest_train); prod_s = prod_s if np.isfinite(prod_s) and prod_s > 1e-6 else 1.0
market_s = city_market_count.std(); market_s = market_s if np.isfinite(market_s) and market_s > 1e-6 else 1.0
prod_z = (city_harvest_train - np.nanmean(city_harvest_train)) / prod_s
market_z = (city_market_count - city_market_count.mean()) / market_s
consumption_proxy_score = market_z - prod_z
consumption_city_mask = consumption_proxy_score >= np.nanquantile(consumption_proxy_score, 0.75)

dow = dates.dayofweek.to_numpy(); doy = dates.dayofyear.to_numpy()
cal = np.stack([np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7), np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)], 1).astype(np.float32)
cal_node = np.repeat(cal[:,None,:], N, 1)

NODE_FEATURE_SLICES = {
    'price': list(range(0, 2)),
    'weather': list(range(2, 2 + 2*len(weather_cols))),
    'supply': list(range(2 + 2*len(weather_cols), 2 + 2*len(weather_cols) + len(agro))),
    'calendar': list(range(2 + 2*len(weather_cols) + len(agro), 2 + 2*len(weather_cols) + len(agro) + 4)),
}
DOMAIN_NAMES = ['price', 'weather', 'supply', 'economy', 'policy']
Xnode = np.concatenate([Zf[...,None], obs[...,None], np.stack(weather,-1), np.stack(weather_anom,-1), np.stack(agro,-1), cal_node], -1).astype(np.float32)
Fn = Xnode.shape[-1]

def z_global(P):
    P = np.asarray(P, float)
    m = np.nanmean(P[istrain], axis=0); s = np.nanstd(P[istrain], axis=0)
    s = np.where(np.isfinite(s) & (s >= 1e-6), s, 1.0)
    return np.nan_to_num((P - m) / s).astype(np.float32)

# Economy: fuel, exchange rate, and leakage-safe BI Food CPI. Ramadan remains a
# known calendar signal. Policy is kept in a separate global domain and never has
# future values supplied to the model.
ECON_NUM_COLS = ['Harga BBM (Pertalite)', 'Harga BBM (Pertamax)', 'Harga BBM (Solar Subsidi)', 'Nilai Kurs', 'food_cpi_mom_pct', 'food_cpi_yoy_pct']
gdf = df.groupby('tanggal')[ECON_NUM_COLS + ['Ramadan']].first().reindex(dates)
econ_num = z_global(gdf[ECON_NUM_COLS].to_numpy(float))
ramadan_flag = gdf['Ramadan'].fillna(0).to_numpy(np.float32).astype(int)
future_ramadan = np.zeros((T, CFG['H']), np.float32)
for h in range(1, CFG['H']+1):
    src = np.arange(T) + h; ok = src < T
    future_ramadan[ok, h-1] = ramadan_flag[src[ok]]
RAMADAN_IDX = econ_num.shape[1]
FUTURE_RAMADAN_SLICE = slice(RAMADAN_IDX + 1, RAMADAN_IDX + 1 + CFG['H'])
G_economy = np.concatenate([econ_num, ramadan_flag[:,None], future_ramadan], axis=1).astype(np.float32)

policy_frame = daily_policy.reindex(dates).fillna(0.0)
G_policy = z_global(policy_frame.to_numpy(float))
GLOBAL_FEATURE_SLICES = {
    'economy': list(range(G_economy.shape[1])),
    'policy': list(range(G_economy.shape[1], G_economy.shape[1] + G_policy.shape[1])),
}
Xglobal = np.concatenate([G_economy, G_policy], axis=1).astype(np.float32)
Fg = Xglobal.shape[1]

mu_t = torch.tensor(mu, dtype=torch.float32); sd_t = torch.tensor(sd, dtype=torch.float32)
print('Xnode', Xnode.shape, '| Fn=', Fn, '| Xglobal', Xglobal.shape, '| Fg=', Fg)
print('domains:', DOMAIN_NAMES, '| policy feature count:', G_policy.shape[1])
print('known-future Ramadan columns:', future_ramadan.shape[1], '| policy has no future columns')
print('production-city proxy:', [cities[i] for i in np.where(production_city_mask)[0]])
print('consumption-city proxy:', [cities[i] for i in np.where(consumption_city_mask)[0]])

Xnode (1826, 120, 26) | Fn= 26 | Xglobal (1826, 60) | Fg= 60
domains: ['price', 'weather', 'supply', 'economy', 'policy'] | policy feature count: 23
known-future Ramadan columns: 30 | policy has no future columns
production-city proxy: ['Kabupaten Banyuwangi', 'Kabupaten Blitar', 'Kabupaten Jember', 'Kabupaten Kediri', 'Kabupaten Lamongan', 'Kabupaten Malang', 'Kabupaten Probolinggo', 'Kabupaten Sampang', 'Kabupaten Situbondo', 'Kabupaten Tuban']
consumption-city proxy: ['Kabupaten Banyuwangi', 'Kabupaten Bondowoso', 'Kabupaten Jember', 'Kabupaten Madiun', 'Kabupaten Mojokerto', 'Kota Batu', 'Kota Kediri', 'Kota Madiun', 'Kota Malang', 'Kota Surabaya']


In [4]:
# === 3b. Freeze the baseline arrays; build spike targets ===
# Everything above this line is the production pipeline, untouched. Scenarios are
# constructed by *adding to* these frozen arrays, so the baseline can never drift.
XNODE_BASE   = Xnode.copy()
XGLOBAL_BASE = Xglobal.copy()
NODE_SLICES_BASE   = {k: list(v) for k, v in NODE_FEATURE_SLICES.items()}
GLOBAL_SLICES_BASE = {k: list(v) for k, v in GLOBAL_FEATURE_SLICES.items()}
DOMAINS_BASE = list(DOMAIN_NAMES)
PRICE_IDX = 0                      # Xnode[...,0] is the normalised log price (Zf)

# --- policy expert -------------------------------------------------------------------
# 'policy' is NOT a news signal. It comes from policy_event_features.csv: 91 JDIH legal
# documents (22 with include_in_model=1), each carrying document_title, document_type,
# legal_number and legal_year - a separate regulation-PDF pipeline.
#
# It is removed from the DEFAULT baseline so that news is the only document-derived
# signal, and its contribution cannot be confounded with a second textual source. The
# block is kept in memory because the S8 scenarios fuse it back in with news.
G_POLICY_BASE = G_policy.copy()
POLICY_COLS = list(POLICY_EVENT_COLS)

DROP_POLICY = True
if DROP_POLICY and 'policy' in GLOBAL_SLICES_BASE:
    _keep = list(GLOBAL_SLICES_BASE['economy'])
    XGLOBAL_BASE = XGLOBAL_BASE[:, _keep].copy()
    GLOBAL_SLICES_BASE = {'economy': list(range(len(_keep)))}
    DOMAINS_BASE = [d for d in DOMAINS_BASE if d != 'policy']
    print(f'policy expert dropped from baseline: Fg {Xglobal.shape[1]} -> {XGLOBAL_BASE.shape[1]}')

# Why fusion is worth a scenario: the regulation record is extremely sparse next to the
# news corpus, and sparse in exactly the place this paper cares about.
_pu = policy_events
_nzp = _pu[POLICY_COLS].astype(float).sum()
print(f'policy events used: {len(_pu)}  span {_pu.event_date.min().date()} .. {_pu.event_date.max().date()}')
print(f'  -> no policy signal at all before {_pu.event_date.min().date()}, '
      f'i.e. the first {int((_pu.event_date.min() - dates[0]).days)} days of the panel')
print(f'  -> chili-tagged policy events: {int(_nzp["chili"])} of {len(_pu)}')
print(f'  -> all-zero policy columns: {[c for c in POLICY_COLS if _nzp[c] == 0]}')

print('baseline frozen: Fn =', XNODE_BASE.shape[-1], ' Fg =', XGLOBAL_BASE.shape[-1],
      ' domains =', DOMAINS_BASE)

# Spike target: |log price(t+h) - log price(t)| > threshold, in Rupiah terms.
# Y is per-node z-scored log price, so the log move is |dY| * sd[node].
THR = float(SCFG['spike_threshold'])
SPIKE = np.zeros((T, N, CFG['H']), np.float32)
for h in range(1, CFG['H'] + 1):
    dlog = np.abs(Y[h:] - Y[:-h]) * sd[None, :]
    SPIKE[:-h, :, h - 1] = (dlog > THR).astype(np.float32)
print('spike rate (|move| > %.0f%%) by horizon:' % (100 * THR),
      {h: round(float(SPIKE[:, :, h - 1].mean()), 4) for h in CFG['HORIZONS']})

policy expert dropped from baseline: Fg 60 -> 37
policy events used: 22  span 2022-09-01 .. 2026-03-11
  -> no policy signal at all before 2022-09-01, i.e. the first 549 days of the panel
  -> chili-tagged policy events: 1 of 22
  -> all-zero policy columns: ['buffalo_meat', 'cooking_oil', 'wheat', 'fish', 'market_operation', 'production']
baseline frozen: Fn = 26  Fg = 37  domains = ['price', 'weather', 'supply', 'economy']
spike rate (|move| > 10%) by horizon: {1: 0.1469, 3: 0.3502, 7: 0.5651, 14: 0.7217, 30: 0.8357}


In [5]:
# === 3c. News channels from news.sqlite ===
# The database itself is not shipped to Colab. build_news_channels.py aggregated it to
# daily channels under strict look-ahead rules (available_date = pub_date + 1 unless the
# article carries an exact timestamp before 06:00 WIB; missing days are zero, never
# back-filled). Body-level annotation throughout - headline and body tagging have
# different coverage and must not be mixed.
NEWS_CHANNELS = ['supply_shock', 'logistics', 'trade_policy', 'production',
                 'supply_reserve', 'chili_focus', 'price_report', 'pressure_net']
NEWS_STRATA = ['all', 'fwd', 'price']

_nat = pd.read_csv('news_channels_national.csv.gz', parse_dates=['available_date'])
_cty = pd.read_csv('news_channels_city.csv.gz', parse_dates=['available_date']).copy()

# Decay is applied over the FULL news history and only then sliced to the price panel.
# Clipping first would start March 2021 from an empty memory and silently discard the
# January-February news that a 30-day half-life should still be carrying.
NEWS_DATES = pd.date_range(min(_nat['available_date'].min(), dates[0]),
                           max(_nat['available_date'].max(), dates[-1]), freq='D')
DATE_POS = NEWS_DATES.get_indexer(dates)
assert (DATE_POS >= 0).all(), 'price-panel dates must lie inside the news date range'

_nat = _nat.set_index('available_date').reindex(NEWS_DATES).fillna(0.0)
_vol = _nat['n_indexed'].to_numpy(float)

_cty = _cty[_cty['city'].isin(cities) & _cty['available_date'].isin(NEWS_DATES)].copy()
_cty['_ci'] = _cty['city'].map(cidx)
_ti = pd.Series(np.arange(len(NEWS_DATES)), index=NEWS_DATES).reindex(
    _cty['available_date']).to_numpy()
_ci = _cty['_ci'].to_numpy()

NEWS_RAW_NAT, NEWS_RAW_CITY = {}, {}
for s in NEWS_STRATA:
    cols = [s + '__' + c for c in NEWS_CHANNELS]
    NEWS_RAW_NAT[s] = _nat[cols].to_numpy(np.float32)                  # (Tn, K)
    arr = np.zeros((len(NEWS_DATES), R, len(NEWS_CHANNELS)), np.float32)
    arr[_ti, _ci, :] = _cty[cols].to_numpy(np.float32)
    NEWS_RAW_CITY[s] = arr                                             # (Tn, R, K)

CITY_OF_NODE = np.array([cidx[node_city[n]] for n in nodes])

def news_decay(x, half_life):
    """Exponential memory. A news shock does not stop mattering the day after it prints."""
    if not half_life or half_life <= 0:
        return x.astype(np.float32)
    lam = 0.5 ** (1.0 / float(half_life))
    out = np.zeros_like(x, dtype=np.float32)
    st = np.zeros(x.shape[1:], np.float32)
    for t in range(x.shape[0]):
        st = lam * st + x[t]
        out[t] = st
    return out

def z_train_only(x):
    """Normalise with TRAIN statistics only. Using full-series statistics would leak the
    test-period distribution into training - the single most common silent error here."""
    m = x[istrain].mean(0); s = x[istrain].std(0)
    s = np.where(np.isfinite(s) & (s > 1e-6), s, 1.0)
    return np.nan_to_num((x - m) / s).astype(np.float32)

def build_news(stratum='all', half_life=None, share_norm=None):
    """-> (national (T,K), per-node (T,N,K)), decayed over the full news history,
    sliced to the price panel, then normalised on train statistics only."""
    hl = SCFG['news_half_life'] if half_life is None else half_life
    sn = SCFG['news_share_norm'] if share_norm is None else share_norm
    nat = NEWS_RAW_NAT[stratum].copy()
    if sn:
        # Indexed volume drifts heavily across 2021-2026; raw counts would encode that
        # platform drift as if it were a change in the news itself.
        nat = nat / np.maximum(_vol, 1.0)[:, None]
    nat = z_train_only(news_decay(nat, hl)[DATE_POS])
    cty = z_train_only(news_decay(NEWS_RAW_CITY[stratum], hl)[DATE_POS])
    return nat, cty[:, CITY_OF_NODE, :]

_n, _c = build_news('all')
print('news channels:', NEWS_CHANNELS)
print('national', _n.shape, ' per-node', _c.shape)
print('strata volume (production channel, raw):',
      {s: float(NEWS_RAW_NAT[s][:, 3].sum()) for s in NEWS_STRATA})
print('share of days with any national signal:',
      round(float((NEWS_RAW_NAT['all'].sum(1) > 0).mean()), 4))
print('share of city-day cells with any signal:',
      round(float((NEWS_RAW_CITY['all'].sum(-1) > 0).mean()), 4))

news channels: ['supply_shock', 'logistics', 'trade_policy', 'production', 'supply_reserve', 'chili_focus', 'price_report', 'pressure_net']
national (1826, 8)  per-node (1826, 120, 8)
strata volume (production channel, raw): {'all': 50291.0, 'fwd': 38893.0, 'price': 11398.0}
share of days with any national signal: 1.0
share of city-day cells with any signal: 0.2146


In [6]:
# === 3d. Policy-news fusion, and per-scenario array assembly ===
# The news ontology was built to mirror the policy taxonomy, so the two vocabularies line
# up directly. Only columns that are actually non-zero in the 22 used events are mapped;
# market_operation and production are all-zero in the regulation record and are left
# alone rather than given a spurious weight.
POLICY_NEWS_ALIGN = {
    'expected_price_pressure':  'pressure_net',     # both signed, same convention
    'chili':                    'chili_focus',
    'price_management':         'trade_policy',
    'import_export':            'trade_policy',
    'supply_reserve':           'supply_reserve',
    'procurement_distribution': 'supply_reserve',
    'subsidy_assistance':       'supply_reserve',
}

def policy_salience(P, nat):
    """Amplify each regulation by the news coverage it actually received.

    A regulation nobody reported on probably did not move a market; one that dominated
    the trade-policy channel for a week plausibly did. `nat` is z-scored on train, so
    relu(nat) is coverage ABOVE the training mean and the factor is >= 1: coverage can
    amplify a policy effect but never flip its sign or erase it.
    """
    out = P.copy()
    hit = []
    for j, col in enumerate(POLICY_COLS):
        ch = POLICY_NEWS_ALIGN.get(col)
        if ch is None:
            continue
        out[:, j] = P[:, j] * (1.0 + np.maximum(nat[:, NEWS_CHANNELS.index(ch)], 0.0))
        hit.append(col)
    return out.astype(np.float32), hit


def scenario_arrays(sc):
    """Return (Xnode, Xglobal, node_slices, global_slices, domain_names) for a scenario.

    Baseline arrays are never mutated; blocks are appended and their column ranges
    recorded so the context encoder can address them.

    Routing rule for national news: a block appended to Xglobal without being named in
    domain_names would reach the model ONLY through CMIL.head_gate - four scalar weights.
    So when news is not given its own expert it is folded into the economy expert's
    slice, where it is actually read. Otherwise 'news as a feature' would be a no-op by
    construction and S1 would test nothing.
    """
    xn, xg = XNODE_BASE, XGLOBAL_BASE
    ns = {k: list(v) for k, v in NODE_SLICES_BASE.items()}
    gs = {k: list(v) for k, v in GLOBAL_SLICES_BASE.items()}
    dom = list(DOMAINS_BASE)

    nat = None
    nw = sc.get('news')
    if nw:
        nat, cty = build_news(nw.get('stratum', 'all'),
                              nw.get('half_life'), nw.get('share_norm'))
        if nw.get('scope', 'both') in ('city', 'both'):
            n0 = xn.shape[-1]
            xn = np.concatenate([xn, cty], axis=-1).astype(np.float32)
            ns['news_node'] = list(range(n0, xn.shape[-1]))
    add_nat = bool(nw) and nw.get('scope', 'both') in ('national', 'both')

    pol = sc.get('policy')
    P = None
    if pol:
        P = G_POLICY_BASE
        if pol == 'salient':
            assert nat is not None, 'salience weighting needs the news block'
            P, _ = policy_salience(G_POLICY_BASE, nat)

    def _append(block, name, as_expert=True):
        nonlocal xg, gs, dom
        g0 = xg.shape[1]
        xg = np.concatenate([xg, block], axis=1).astype(np.float32)
        rng = list(range(g0, xg.shape[1]))
        if as_expert:
            gs[name] = rng
            dom = dom + [name]
        else:
            gs['economy'] = gs['economy'] + rng      # read by the economy expert
        return rng

    if sc.get('fuse_expert'):
        # S8 fused: ONE joint expert over [policy || news]. The two sources are seen
        # together rather than gated against each other.
        assert P is not None and add_nat, 'fuse_expert needs both policy and national news'
        _append(np.concatenate([P, nat], axis=1), 'polnews')
    else:
        if P is not None:
            _append(P, 'policy')
        if add_nat:
            _append(nat, 'news', as_expert=bool(sc.get('news_expert')))
    return xn, xg, ns, gs, dom


_probe = [('baseline', {}),
          ('S1 news (feature)', {'news': {'stratum': 'all'}}),
          ('S5 news (expert)', {'news': {'stratum': 'all'}, 'news_expert': True}),
          ('S8 policy only', {'policy': 'raw'}),
          ('S8 fused', {'news': {'stratum': 'all'}, 'policy': 'raw', 'fuse_expert': True}),
          ('S8 salience', {'news': {'stratum': 'all'}, 'policy': 'salient', 'fuse_expert': True})]
for _name, _sc in _probe:
    a, b, _, _, d = scenario_arrays(_sc)
    print(f'{_name:20s} Fn={a.shape[-1]:3d}  Fg={b.shape[1]:3d}  domains={d}')
_, _hit = policy_salience(G_POLICY_BASE, build_news('all')[0])
print(f'\nsalience-weighted policy columns ({len(_hit)} of {len(POLICY_COLS)}): {_hit}')

baseline             Fn= 26  Fg= 37  domains=['price', 'weather', 'supply', 'economy']
S1 news (feature)    Fn= 34  Fg= 45  domains=['price', 'weather', 'supply', 'economy']
S5 news (expert)     Fn= 34  Fg= 45  domains=['price', 'weather', 'supply', 'economy', 'news']
S8 policy only       Fn= 26  Fg= 60  domains=['price', 'weather', 'supply', 'economy', 'policy']
S8 fused             Fn= 34  Fg= 68  domains=['price', 'weather', 'supply', 'economy', 'polnews']
S8 salience          Fn= 34  Fg= 68  domains=['price', 'weather', 'supply', 'economy', 'polnews']

salience-weighted policy columns (7 of 23): ['expected_price_pressure', 'chili', 'price_management', 'supply_reserve', 'procurement_distribution', 'import_export', 'subsidy_assistance']


In [7]:
# === 5. Model (Path B): Multi-scale Dilated Temporal Encoder + per-market Context Attribution ===
class DilatedTemporalEncoder(nn.Module):
    """Multi-scale dilated temporal encoder (Graph WaveNet philosophy, no adaptive graph).

    Per-node sequence -> stacked causal dilated conv blocks (gated activation, residual,
    skip) -> market representation at the final step. Replaces the LSTM market encoder.
    """
    def __init__(self, fn, d, dilations=(1, 2, 4, 8, 16), kernel=2):
        super().__init__()
        self.dilations, self.kernel = dilations, kernel
        self.input_proj = nn.Conv1d(fn, d, 1)
        self.filt = nn.ModuleList(nn.Conv1d(d, d, kernel, dilation=dl) for dl in dilations)
        self.gate = nn.ModuleList(nn.Conv1d(d, d, kernel, dilation=dl) for dl in dilations)
        self.resid = nn.ModuleList(nn.Conv1d(d, d, 1) for _ in dilations)
        self.skipc = nn.ModuleList(nn.Conv1d(d, d, 1) for _ in dilations)
        self.out = nn.Sequential(nn.ReLU(), nn.Conv1d(d, d, 1))

    def forward(self, x):                              # x: (B*N, L, Fn)
        x = self.input_proj(x.transpose(1, 2))         # (B*N, d, L)
        skip = 0
        for i, dl in enumerate(self.dilations):
            xp = F.pad(x, ((self.kernel - 1) * dl, 0))                 # causal left pad
            z = torch.tanh(self.filt[i](xp)) * torch.sigmoid(self.gate[i](xp))  # gated, (B*N,d,L)
            skip = skip + self.skipc[i](z)
            x = x + self.resid[i](z)                                   # residual
        return self.out(skip)[:, :, -1]                # (B*N, d) at last step


class ContextEncoder(nn.Module):
    """Five-domain context encoder: price, weather, supply, economy, and policy."""
    def __init__(self, fn, fg, dc, node_slices, global_slices, domain_names):
        super().__init__()
        self.node_slices, self.global_slices, self.domain_names = node_slices, global_slices, domain_names
        self.price = nn.Linear(len(node_slices['price']), dc)
        self.weather = nn.Linear(len(node_slices['weather']), dc)
        self.supply = nn.Linear(len(node_slices['supply']), dc)
        self.economy = nn.Linear(len(global_slices['economy']), dc)
        self.policy = nn.Linear(len(global_slices['policy']), dc)
        self.weight_head = nn.Sequential(nn.Linear(len(domain_names) * dc, dc), nn.ReLU(), nn.Linear(dc, len(domain_names)))

    def _node_summary(self, xn, key):
        return xn[..., self.node_slices[key]].mean(dim=(1, 2))

    def forward(self, xn, xg):
        ep = torch.tanh(self.price(self._node_summary(xn, 'price')))
        ew = torch.tanh(self.weather(self._node_summary(xn, 'weather')))
        es = torch.tanh(self.supply(self._node_summary(xn, 'supply')))
        ee = torch.tanh(self.economy(xg[:, self.global_slices['economy']]))
        e_policy = torch.tanh(self.policy(xg[:, self.global_slices['policy']]))
        E = torch.stack([ep, ew, es, ee, e_policy], dim=1)
        ctx_weight = F.softmax(self.weight_head(E.reshape(E.shape[0], -1)), dim=-1)
        C = (ctx_weight.unsqueeze(-1) * E).sum(dim=1)
        return C, ctx_weight, E

class NodeContextAttention(nn.Module):
    """Per-market context attribution. Each market forms a query from (H_i, C_t) and attends
    over the factor experts, so different markets weight environment / supply / economy /
    history differently (e.g. Surabaya -> economy, Banyuwangi -> weather). The attention is
    load-bearing: its output modulates the market representation."""
    def __init__(self, d, dc):
        super().__init__()
        self.q   = nn.Sequential(nn.Linear(d + dc, d), nn.ReLU(), nn.Linear(d, dc))
        self.key = nn.Linear(dc, dc)
        self.val = nn.Linear(dc, d)

    def forward(self, H, C, E):                        # H:(B,N,d) C:(B,dc) E:(B,5,dc)
        B, N, _ = H.shape
        q = self.q(torch.cat([H, C.unsqueeze(1).expand(-1, N, -1)], -1))  # (B,N,dc)
        K, V = self.key(E), self.val(E)                                   # (B,5,dc),(B,5,d)
        att = F.softmax(torch.einsum('bnd,bed->bne', q, K) / (q.shape[-1] ** 0.5), -1)  # (B,N,5)
        ctx = torch.einsum('bne,bed->bnd', att, V)                        # (B,N,d) per-market context
        return ctx, att


class CMIL(nn.Module):
    """City-level interaction graph (kept from the stable design): learnable city identities,
    context-weighted relation heads, alpha floor anchoring A_phys. The learned structure is
    stable across regimes -- that stability is now a reported finding, not a target to defeat."""
    def __init__(self, d, dc, R, Fg, n_heads=4, context_alpha=True):
        super().__init__()
        assert d % n_heads == 0, 'd must be divisible by n_heads'
        self.d, self.R, self.n_heads, self.dh = d, R, n_heads, d // n_heads
        self.context_alpha = context_alpha
        self.city_emb = nn.Parameter(torch.randn(R, d) * 0.1)
        self.gamma = nn.Linear(dc, d); self.beta = nn.Linear(dc, d)
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wv = nn.Linear(d, d, bias=False)
        self.head_gate = nn.Linear(dc + Fg, n_heads)
        self.context_edge_bias = nn.Sequential(nn.Linear(dc, d), nn.Tanh(), nn.Linear(d, 1))
        if context_alpha:
            self.alpha_head = nn.Sequential(nn.Linear(dc, dc), nn.ReLU(), nn.Linear(dc, 1))
        else:
            self.alpha_logit = nn.Parameter(torch.tensor(0.0))
        self.alpha_min, self.alpha_span = 0.2, 0.6

    def forward(self, H, Sbar, C, Aphys, xg):
        B = H.shape[0]
        G  = torch.einsum('nr,bnd->brd', Sbar, H) + self.city_emb.unsqueeze(0)
        Gt = self.gamma(C).unsqueeze(1) * G + self.beta(C).unsqueeze(1)
        Q = self.Wq(Gt).view(B, self.R, self.n_heads, self.dh).transpose(1, 2)
        K = self.Wk(Gt).view(B, self.R, self.n_heads, self.dh).transpose(1, 2)
        V = self.Wv(G)
        Ih = torch.matmul(Q, K.transpose(-1, -2)) / (self.dh ** 0.5)
        head_w = F.softmax(self.head_gate(torch.cat([C, xg], dim=-1)), dim=-1)
        I = (head_w.view(B, self.n_heads, 1, 1) * Ih).sum(1)
        temp = 0.5 + 0.5 * torch.sigmoid(self.context_edge_bias(C).view(-1, 1, 1))
        Actx = F.softmax(I / temp, dim=-1)
        raw = (torch.sigmoid(self.alpha_head(C)).view(-1, 1, 1) if self.context_alpha
               else torch.sigmoid(self.alpha_logit))
        alpha = self.alpha_min + self.alpha_span * raw
        At = alpha * Aphys.unsqueeze(0) + (1.0 - alpha) * Actx
        return torch.matmul(At, V), At, {'Actx': Actx, 'alpha': alpha, 'temperature': temp, 'head_w': head_w}


class CMIN(nn.Module):
    def __init__(self, Fn, Fg, d, dc, H):
        super().__init__()
        self.enc       = DilatedTemporalEncoder(Fn, d)
        self.ctx       = ContextEncoder(Fn, Fg, dc, NODE_FEATURE_SLICES, GLOBAL_FEATURE_SLICES, DOMAIN_NAMES)
        self.node_attn = NodeContextAttention(d, dc)
        self.cmil      = CMIL(d, dc, R, Fg, n_heads=CFG.get('n_heads', 4),
                              context_alpha=CFG.get('context_alpha', True))
        self.gate = nn.Linear(2 * d, d)
        self.gate_min, self.gate_span = 0.15, 0.70
        self.dec  = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))

    def forward(self, xn, xg, Sbar, S, Aphys):
        B, N, L, Fn = xn.shape
        h = self.enc(xn.reshape(B * N, L, Fn)).reshape(B, N, -1)          # dilated temporal encoder
        C, ctx_weight, E = self.ctx(xn, xg)
        node_ctx, node_att = self.node_attn(h, C, E)                      # per-market context attribution
        h = h + node_ctx                                                 # context modulates the market repr
        Gc, At, aux = self.cmil(h, Sbar, C, Aphys, xg)
        back = torch.einsum('nr,brd->bnd', S, Gc)
        g = self.gate_min + self.gate_span * torch.sigmoid(self.gate(torch.cat([h, back], -1)))
        z = g * h + (1.0 - g) * back
        aux['ctx_weight'] = ctx_weight                                   # global domain weights
        aux['node_ctx_attention'] = node_att                            # (B,N,5) per-market attribution
        aux['domain_names'] = DOMAIN_NAMES
        return self.dec(z), At, aux


print('Path-B model: dilated temporal encoder + per-market context attribution (stable interaction graph)')

Path-B model: dilated temporal encoder + per-market context attribution (stable interaction graph)


In [8]:
# === 5b. Scenario-aware model ===
# DilatedTemporalEncoder, NodeContextAttention and CMIL are reused unchanged from the
# production model above. Only the context encoder and the output head vary.

def seq_summary(xg_seq):
    """(B,L,Fg) -> mean, last, slope. S7 replaces the context snapshot with this."""
    B, Lw, _ = xg_seq.shape
    tt = torch.linspace(-1.0, 1.0, Lw, device=xg_seq.device).view(1, Lw, 1)
    mean = xg_seq.mean(1)
    last = xg_seq[:, -1]
    slope = (xg_seq * tt).mean(1) / (tt ** 2).mean()
    return mean, last, slope


class ContextEncoderS(nn.Module):
    """Context encoder whose experts are built FROM domain_names, not hard-coded.

    Each name in domain_names must resolve to either a node slice (price, weather,
    supply) or a global slice (economy, news, and policy when enabled). Dropping or
    adding a domain is then a data-side edit only - no surgery on this class. S7 feeds
    each global expert mean/last/slope over the window instead of the value at t,
    tripling its input width.
    """
    def __init__(self, dc, node_slices, global_slices, domain_names, ctx_seq=False):
        super().__init__()
        self.node_slices, self.global_slices = node_slices, global_slices
        self.domain_names, self.ctx_seq = list(domain_names), ctx_seq
        mult = 3 if ctx_seq else 1
        self.proj = nn.ModuleDict()
        for k in self.domain_names:
            if k in node_slices:
                self.proj[k] = nn.Linear(len(node_slices[k]), dc)
            elif k in global_slices:
                self.proj[k] = nn.Linear(mult * len(global_slices[k]), dc)
            else:
                raise KeyError(f'domain {k!r} has no node or global slice')
        self.weight_head = nn.Sequential(
            nn.Linear(len(self.domain_names) * dc, dc), nn.ReLU(),
            nn.Linear(dc, len(self.domain_names)))

    def _node_summary(self, xn, key):
        return xn[..., self.node_slices[key]].mean(dim=(1, 2))

    def _g(self, parts, key):
        return torch.cat([p[:, self.global_slices[key]] for p in parts], dim=-1)

    def forward(self, xn, xg_seq):
        parts = seq_summary(xg_seq) if self.ctx_seq else (xg_seq[:, -1],)
        E = []
        for k in self.domain_names:                      # order defines ctx_weight order
            src = self._node_summary(xn, k) if k in self.node_slices else self._g(parts, k)
            E.append(torch.tanh(self.proj[k](src)))
        E = torch.stack(E, dim=1)
        ctx_weight = F.softmax(self.weight_head(E.reshape(E.shape[0], -1)), dim=-1)
        C = (ctx_weight.unsqueeze(-1) * E).sum(dim=1)
        return C, ctx_weight, E


class CMINScenario(nn.Module):
    def __init__(self, Fn_, Fg_, d, dc, H_, node_slices, global_slices, domain_names, sc):
        super().__init__()
        self.sc = sc
        self.residual = bool(sc.get('residual_head'))
        self.spike = bool(sc.get('spike_head'))
        self.enc = DilatedTemporalEncoder(Fn_, d)
        self.ctx = ContextEncoderS(dc, node_slices, global_slices, domain_names,
                                   ctx_seq=bool(sc.get('ctx_seq')))
        self.node_attn = NodeContextAttention(d, dc)
        self.cmil = CMIL(d, dc, R, Fg_, n_heads=CFG.get('n_heads', 4),
                         context_alpha=CFG.get('context_alpha', True))
        self.gate = nn.Linear(2 * d, d)
        self.gate_min, self.gate_span = 0.15, 0.70
        self.dec = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H_))
        self.spike_head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H_)) if self.spike else None

    def forward(self, xn, xg_seq, Sbar, S_, Aphys):
        B, Nn, Lw, _ = xn.shape
        h = self.enc(xn.reshape(B * Nn, Lw, -1)).reshape(B, Nn, -1)
        C, ctx_weight, E = self.ctx(xn, xg_seq)
        node_ctx, node_att = self.node_attn(h, C, E)
        h = h + node_ctx
        Gc, At, aux = self.cmil(h, Sbar, C, Aphys, xg_seq[:, -1])
        back = torch.einsum('nr,brd->bnd', S_, Gc)
        g = self.gate_min + self.gate_span * torch.sigmoid(self.gate(torch.cat([h, back], -1)))
        z = g * h + (1.0 - g) * back

        out = self.dec(z)
        if self.residual:
            # S0: predict the DEVIATION from the last observed price. Without this the
            # model must reconstruct an absolute level from a 64-d bottleneck, which is
            # exactly the information a compressed representation has discarded - and is
            # why the level-predicting head loses to persistence at h=1.
            out = xn[:, :, -1, PRICE_IDX].unsqueeze(-1) + out

        aux['ctx_weight'] = ctx_weight
        aux['node_ctx_attention'] = node_att
        aux['domain_names'] = self.ctx.domain_names
        aux['spike_logits'] = self.spike_head(z) if self.spike else None
        return out, At, aux

print('scenario model ready')

scenario model ready


In [9]:
# === 5c. v3 model: expert representations exposed, REST-TS head, news ablation switch ===
class CMINv3(CMINScenario):
    """Adds three things to the v2 model:
      1. aux['E'] - per-sample expert representations, needed for effective rank.
         v2 stored only the batch-MEAN context weight, which averages away exactly the
         variation effective rank measures. That is why v2 cannot answer this offline.
      2. an optional REST-TS head: a news-only decoder supervised on the numerical
         residual, so no numerical pathway can reduce its loss.
      3. news_scale - multiplies the news block at inference, for the divergence test.
    """
    def __init__(self, Fn_, Fg_, d, dc, H_, node_slices, global_slices, domain_names, sc):
        super().__init__(Fn_, Fg_, d, dc, H_, node_slices, global_slices, domain_names, sc)
        self.rest = bool(sc.get('rest'))
        self.news_idx = domain_names.index('news') if 'news' in domain_names else None
        if self.rest:
            assert self.news_idx is not None, 'REST needs a news expert'
            self.dec_news = nn.Sequential(nn.Linear(dc, d), nn.ReLU(), nn.Linear(d, H_))

    def forward(self, xn, xg_seq, Sbar, S_, Aphys):
        out, At, aux = super().forward(xn, xg_seq, Sbar, S_, Aphys)
        _, _, E = self.ctx(xn, xg_seq)
        aux['E'] = E                                   # (B, n_experts, dc)
        if self.rest:
            e_news = E[:, self.news_idx, :]            # (B, dc) - news pathway ONLY
            pred_news = self.dec_news(e_news).unsqueeze(1).expand(-1, xn.shape[1], -1)
            aux['pred_num'] = out                      # numerical backbone forecast
            aux['pred_news'] = pred_news               # residual correction
            out = out + pred_news
        return out, At, aux


def zero_news(xn, xg, ns, gs):
    """Return copies with every news column set to 0 - the divergence probe."""
    xn2, xg2 = xn.clone(), xg.clone()
    if 'news' in gs:      xg2[..., gs['news']] = 0.0
    if 'polnews' in gs:   xg2[..., gs['polnews']] = 0.0
    if 'news_node' in ns: xn2[..., ns['news_node']] = 0.0
    return xn2, xg2


def effective_rank(Z, eps=1e-12):
    """Roy & Vetterli effective rank: exp(Shannon entropy of the normalised spectrum).

    High  -> representations spread out across inputs (content-dependent).
    ~1    -> every input maps to nearly the same direction (COLLAPSED).
    """
    Z = Z - Z.mean(0, keepdims=True)
    s = np.linalg.svd(Z, compute_uv=False)
    p = s / max(s.sum(), eps)
    p = p[p > eps]
    return float(np.exp(-(p * np.log(p)).sum()))

print('v3 model ready: aux[E] exposed, REST head available, news-zeroing probe defined')

v3 model ready: aux[E] exposed, REST head available, news-zeroing probe defined


In [10]:
# === 4b. Windowing (scenario-aware) ===
L, H = CFG['L'], CFG['H']
REGIME_H = min(int(CFG.get('regime_horizon', 7)), H)
origins = list(range(L - 1, T - H))
def tgt_start(t): return dates[t + 1]
test_o = [t for t in origins if tgt_start(t) >= TEST_START]
pool   = [t for t in origins if tgt_start(t) <  TEST_START]
nval   = max(30, int(0.1 * len(pool)))
train_o, val_o = pool[:-nval], pool[-nval:]
print(f'windows -> train={len(train_o)}  val={len(val_o)}  test={len(test_o)}')

YY = torch.tensor(Y); OB = torch.tensor(obs); SPK = torch.tensor(SPIKE)
Sbar_t = torch.tensor(Sbar).to(DEVICE)
S_t    = torch.tensor(S).to(DEVICE)
Aphys_t = torch.tensor(A_phys).to(DEVICE)

class WDS(Dataset):
    """Always returns the global context as a WINDOW. Non-S7 scenarios take [:, -1],
    so a single dataset serves every scenario and the sampling can never differ."""
    def __init__(self, oo, xn, xg):
        self.oo = oo
        self.xn = torch.tensor(xn); self.xg = torch.tensor(xg)
    def __len__(self): return len(self.oo)
    def __getitem__(self, k):
        t = self.oo[k]
        return (self.xn[t - L + 1:t + 1].permute(1, 0, 2),   # (N,L,Fn)
                self.xg[t - L + 1:t + 1],                    # (L,Fg)
                YY[t + 1:t + 1 + H].permute(1, 0),           # (N,H)
                OB[t + 1:t + 1 + H].permute(1, 0),           # (N,H)
                SPK[t],                                      # (N,H)
                YY[t])                                       # (N,) persistence anchor

def make_loader_s(oo, sh, xn, xg):
    return DataLoader(WDS(oo, xn, xg), batch_size=CFG['batch'], shuffle=sh)

# Horizon weights for S0. h=1 errors are an order of magnitude smaller than h=30 errors,
# so a uniformly averaged loss spends almost none of its gradient on the short horizons.
_tr = np.array(train_o)
_scale = np.zeros(H, np.float64)
for h in range(1, H + 1):
    d_ = np.abs(Y[_tr + h] - Y[_tr]) * obs[_tr + h]
    _scale[h - 1] = d_.sum() / max(obs[_tr + h].sum(), 1)
HW = 1.0 / np.maximum(_scale, 1e-6)
HW = np.clip(HW / HW.mean(), 1.0 / SCFG['horizon_weight_clip'], SCFG['horizon_weight_clip'])
HW_t = torch.tensor(HW, dtype=torch.float32, device=DEVICE)
print('per-horizon |dY| scale  h1..h5:', np.round(_scale[:5], 4))
print('horizon weights         h1..h5:', np.round(HW[:5], 3), ' h30:', round(float(HW[-1]), 3))

windows -> train=1287  val=143  test=337
per-horizon |dY| scale  h1..h5: [0.1056 0.1796 0.237  0.2847 0.3285]
horizon weights         h1..h5: [4.485 2.637 1.998 1.663 1.442]  h30: 0.485


In [11]:
# === 6b. Train / evaluate, with per-seed retention and early-stop instrumentation ===
def masked_mae_w(p, y, m, w=None):
    e = torch.abs(p - y) * m
    if w is None:
        return e.sum() / m.sum().clamp(min=1)
    w = w.view(1, 1, -1)
    return (e * w).sum() / (m * w).sum().clamp(min=1)

def graph_sparsity_penalty_s(aux):
    lam = float(CFG.get('lambda_graph_sparsity', 0.03))
    if not lam or 'Actx' not in aux:
        return 0.0
    A = aux['Actx'].clamp_min(1e-9)
    return lam * (-(A * A.log()).sum(dim=-1).mean())

def train_scenario(sc, seed, epochs=None, patience=None, verbose=False):
    """Returns (model, arrays, info). `info` records where early stopping landed, so a
    run that hit the epoch cap - i.e. was still improving when cut off - is visible
    rather than silently truncated."""
    epochs = int(epochs or SCFG['epochs']); patience = int(patience or SCFG['patience'])
    xn, xg, ns, gs, dom = scenario_arrays(sc)
    torch.manual_seed(seed); np.random.seed(seed)
    model = CMINScenario(xn.shape[-1], xg.shape[1], CFG['d'], CFG['d_ctx'], H,
                         ns, gs, dom, sc).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=min(CFG['lr'], 5e-4),
                           weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
    tl = make_loader_s(train_o, True, xn, xg)
    vl = make_loader_s(val_o, False, xn, xg)
    hw = HW_t if sc.get('horizon_weight') else None
    lam_spk = float(SCFG['lambda_spike']) if sc.get('spike_head') else 0.0

    best, best_state, best_ep, wait, hist = 1e9, None, 0, 0, []
    ep = 0
    for ep in range(epochs):
        model.train()
        for xnb, xgb, y, m, spk, _ in tl:
            xnb, xgb = xnb.to(DEVICE), xgb.to(DEVICE)
            y, m, spk = y.to(DEVICE), m.to(DEVICE), spk.to(DEVICE)
            opt.zero_grad()
            p, _, aux = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
            loss = masked_mae_w(p, y, m, hw) + graph_sparsity_penalty_s(aux)
            if lam_spk and aux['spike_logits'] is not None:
                bce = F.binary_cross_entropy_with_logits(aux['spike_logits'], spk, reduction='none')
                loss = loss + lam_spk * ((bce * m).sum() / m.sum().clamp(min=1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()

        model.eval(); vs = vn = 0.0
        with torch.no_grad():
            for xnb, xgb, y, m, spk, _ in vl:
                xnb, xgb, y, m = xnb.to(DEVICE), xgb.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
                p, _, _ = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
                vs += (torch.abs(p - y) * m).sum().item(); vn += m.sum().item()
        v = vs / max(vn, 1); sched.step(v); hist.append(v)
        if verbose:
            print(f'    ep {ep+1:02d} val={v:.4f}')
        if v < best - 1e-4:
            best, best_ep, wait = v, ep + 1, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    info = dict(best_epoch=best_ep, epochs_run=ep + 1, cap=epochs,
                capped=bool(ep + 1 >= epochs), best_val=best, history=hist)
    return model, (xn, xg), info

@torch.no_grad()
def predict_scenario(model, oo, xn, xg):
    model.eval(); P=[]; Yt=[]; Mt=[]; Sp=[]; Lg=[]; An=[]
    for xnb, xgb, y, m, spk, anchor in make_loader_s(oo, False, xn, xg):
        p, _, aux = model(xnb.to(DEVICE), xgb.to(DEVICE), Sbar_t, S_t, Aphys_t)
        P.append(p.cpu()); Yt.append(y); Mt.append(m); Sp.append(spk); An.append(anchor)
        Lg.append(aux['spike_logits'].cpu() if aux['spike_logits'] is not None else torch.zeros_like(p.cpu()))
    return (torch.cat(P), torch.cat(Yt), torch.cat(Mt), torch.cat(Sp),
            torch.cat(Lg), torch.cat(An))

def to_rp(z):
    return torch.exp(z * sd_t.view(1, -1, 1) + mu_t.view(1, -1, 1))

def eval_scenario(model, oo, xn, xg):
    P, Yv, Mv, Spk, Lg, An = predict_scenario(model, oo, xn, xg)
    Prp, Yrp = to_rp(P), to_rp(Yv)
    Perp = to_rp(An.unsqueeze(-1).expand(-1, -1, CFG['H']))
    rows, errs = [], {}
    for h in CFG['HORIZONS']:
        i = h - 1
        mm = Mv[:, :, i] > 0
        pe, ye, pp = Prp[:, :, i][mm], Yrp[:, :, i][mm], Perp[:, :, i][mm]
        sp = Spk[:, :, i][mm] > 0
        ae, ap = torch.abs(pe - ye), torch.abs(pp - ye)
        row = dict(h=h, MAE_Rp=float(ae.mean()),
                   RMSE_Rp=float(torch.sqrt(((pe - ye) ** 2).mean())),
                   MAPE_pct=float((ae / ye.clamp(min=1)).mean() * 100),
                   MAE_persistence=float(ap.mean()), n=int(mm.sum()))
        row['skill_vs_persistence_pct'] = 100.0 * (1.0 - row['MAE_Rp'] / max(row['MAE_persistence'], 1e-9))
        if sp.any():
            row['MAE_spike'] = float(ae[sp].mean())
            row['spike_n'] = int(sp.sum())
        if model.spike:
            pr = torch.sigmoid(Lg[:, :, i][mm]); y01 = sp.float()
            if 0 < y01.mean() < 1:
                order = torch.argsort(pr)
                ranks = torch.empty_like(pr); ranks[order] = torch.arange(len(pr), dtype=pr.dtype) + 1
                npos, nneg = y01.sum(), (1 - y01).sum()
                row['spike_AUC'] = float((ranks[y01 > 0].sum() - npos * (npos + 1) / 2) / (npos * nneg))
        rows.append(row)
        # per-horizon (window, node) absolute errors, retained per seed
        errs[h] = (torch.abs(Prp[:, :, i] - Yrp[:, :, i]) * (Mv[:, :, i] > 0)).numpy()
    return pd.DataFrame(rows).set_index('h'), errs

print('harness ready (per-seed retention + early-stop instrumentation)')

harness ready (per-seed retention + early-stop instrumentation)


In [12]:
# === 6b. Training with HORIZON-NORMALISED selection ===
def masked_mae_w(p, y, m, w=None):
    e = torch.abs(p - y) * m
    if w is None:
        return e.sum() / m.sum().clamp(min=1)
    w = w.view(1, 1, -1)
    return (e * w).sum() / (m * w).sum().clamp(min=1)

def graph_sparsity_penalty_s(aux):
    lam = float(CFG.get('lambda_graph_sparsity', 0.03))
    if not lam or 'Actx' not in aux: return 0.0
    A = aux['Actx'].clamp_min(1e-9)
    return lam * (-(A * A.log()).sum(dim=-1).mean())

# per-horizon error scale from the TRAIN split - the same vector that defines HW
HSCALE = torch.tensor(np.maximum(_scale, 1e-6), dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def val_score(model, loader):
    """Horizon-NORMALISED validation MAE.

    v2 averaged raw MAE over H=30. Long-horizon errors are ~10x larger, so selection was
    effectively h=30-only and fired at epochs arbitrary for h=1 - producing an ablation
    table that ranked by stopping epoch (corr +0.818). Dividing each horizon by its own
    train-residual scale gives every horizon equal say.
    """
    num = torch.zeros(H, device=DEVICE); den = torch.zeros(H, device=DEVICE)
    for xnb, xgb, y, m, spk, _ in loader:
        xnb, xgb, y, m = xnb.to(DEVICE), xgb.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        p, _, _ = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
        num += ((p - y).abs() * m).sum(dim=(0, 1)); den += m.sum(dim=(0, 1)).clamp(min=1)
    return float(((num / den) / HSCALE).mean())

def train_scenario(sc, seed, epochs=None, patience=None, verbose=False):
    epochs = int(epochs or SCFG['epochs']); patience = int(patience or SCFG['patience'])
    xn, xg, ns, gs, dom = scenario_arrays(sc)
    torch.manual_seed(seed); np.random.seed(seed)
    model = CMINv3(xn.shape[-1], xg.shape[1], CFG['d'], CFG['d_ctx'], H, ns, gs, dom, sc).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=min(CFG['lr'], 5e-4), weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
    tl = make_loader_s(train_o, True, xn, xg); vl = make_loader_s(val_o, False, xn, xg)
    hw = HW_t if sc.get('horizon_weight') else None
    lam_rest = float(CFG.get('lambda_rest', 1.0)) if sc.get('rest') else 0.0

    best, best_state, best_ep, wait, hist = 1e9, None, 0, 0, []
    ep = 0
    for ep in range(epochs):
        model.train()
        for xnb, xgb, y, m, spk, _ in tl:
            xnb, xgb = xnb.to(DEVICE), xgb.to(DEVICE)
            y, m = y.to(DEVICE), m.to(DEVICE)
            opt.zero_grad()
            p, _, aux = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
            if lam_rest:
                # REST-TS: the numerical backbone is supervised on y; the news branch is
                # supervised ONLY on the residual it leaves behind. Detaching the residual
                # means no numerical pathway can reduce this term - the news branch has to
                # find content in the text or contribute nothing.
                pn = aux['pred_num']
                loss = masked_mae_w(pn, y, m, hw)
                loss = loss + lam_rest * masked_mae_w(aux['pred_news'], (y - pn).detach(), m, hw)
            else:
                loss = masked_mae_w(p, y, m, hw)
            loss = loss + graph_sparsity_penalty_s(aux)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()

        model.eval(); v = val_score(model, vl); sched.step(v); hist.append(v)
        if verbose: print(f'    ep {ep+1:02d} val_norm={v:.4f}')
        if v < best - 1e-5:
            best, best_ep, wait = v, ep + 1, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, (xn, xg, ns, gs, dom), dict(best_epoch=best_ep, epochs_run=ep + 1,
                                              capped=bool(ep + 1 >= epochs),
                                              best_val=best, history=hist)

@torch.no_grad()
def instrument(model, arrays, oo):
    """effective rank per expert + divergence delta from zeroing news on the SAME model."""
    xn, xg, ns, gs, dom = arrays
    model.eval()
    Es, Pf, Pz, Y, M = [], [], [], [], []
    for xnb, xgb, y, m, spk, _ in make_loader_s(oo, False, xn, xg):
        xnb, xgb = xnb.to(DEVICE), xgb.to(DEVICE)
        p, _, aux = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
        Es.append(aux['E'].cpu().numpy()); Pf.append(p.cpu()); Y.append(y); M.append(m)
        xz, gz = zero_news(xnb, xgb, ns, gs)
        pz, _, _ = model(xz, gz, Sbar_t, S_t, Aphys_t)
        Pz.append(pz.cpu())
    E = np.concatenate(Es, 0)                       # (N, n_experts, dc)
    er = {d: effective_rank(E[:, i, :]) for i, d in enumerate(dom)}
    Pf, Pz = torch.cat(Pf), torch.cat(Pz)
    Yv, Mv = torch.cat(Y), torch.cat(M)
    Prp, Zrp, Yrp = to_rp(Pf), to_rp(Pz), to_rp(Yv)
    dl = {}
    for h in CFG['HORIZONS']:
        i = h - 1; mm = Mv[:, :, i] > 0
        num = torch.sqrt((((Prp[:, :, i] - Zrp[:, :, i]) ** 2)[mm]).mean())
        den = torch.abs(Prp[:, :, i] - Yrp[:, :, i])[mm].mean()
        dl[h] = float(num / den.clamp(min=1e-9))
    return er, dl

print('horizon-normalised selection active; instrumentation ready')

horizon-normalised selection active; instrumentation ready


In [13]:
# === 7c. Placebo transforms (v4: any auxiliary block, not just news) ===
# v3 scrambled only the news columns, so policy could never be placebo-tested. Here the
# target is explicit, and defaults to every auxiliary block present.
_rng_placebo = np.random.default_rng(12345)
AUX_GLOBAL = ('news', 'policy', 'polnews')
AUX_NODE = ('news_node',)

def _targets(sc, ns, gs):
    want = sc.get('placebo_on')
    want = (want,) if isinstance(want, str) else (want or AUX_GLOBAL + AUX_NODE)
    return ([k for k in AUX_GLOBAL if k in gs and k in want],
            [k for k in AUX_NODE if k in ns and k in want])

def apply_placebo(sc, xn, xg, ns, gs):
    kind = sc.get('placebo')
    if not kind:
        return xn, xg
    gk, nk = _targets(sc, ns, gs)
    xn, xg = xn.copy(), xg.copy()
    if kind == 'time':
        # break the block's correspondence to days; volume and marginals preserved
        perm = _rng_placebo.permutation(xg.shape[0])
        for k in gk: xg[:, gs[k]] = xg[perm][:, gs[k]]
        for k in nk: xn[..., ns[k]] = xn[perm][..., ns[k]]
    elif kind == 'city':
        # keep the timing, scramble WHICH market receives each story
        perm = _rng_placebo.permutation(xn.shape[1])
        for k in nk: xn[..., ns[k]] = xn[:, perm][..., ns[k]]
        if not nk:
            print('  [!] city placebo requested but this scenario has no node-level block')
    return xn, xg

_orig_scenario_arrays = scenario_arrays
def scenario_arrays(sc):                       # noqa: F811 - deliberate wrapper
    xn, xg, ns, gs, dom = _orig_scenario_arrays(sc)
    xn, xg = apply_placebo(sc, xn, xg, ns, gs)
    return xn, xg, ns, gs, dom

print('placebo wrapper installed (targets:', AUX_GLOBAL + AUX_NODE, ')')

placebo wrapper installed (targets: ('news', 'policy', 'polnews', 'news_node') )


In [14]:
# === P1. Ablation ladder: graph mode as an explicit switch ===
# The ladder needs adjacency modes that CMIL does not expose, so they are added by subclass
# here rather than by editing the copied v4 cells. Everything else -- temporal encoder,
# context encoder, fusion gate, head, loss -- is untouched, which is the whole point: the
# rungs must differ in the graph and in nothing else.
SMOKE = True          # True: 1 seed, 3 epochs, every model. Set False for the real run.

PROTO_DIR = os.path.join(OUT_DIR, 'protocol')
os.makedirs(PROTO_DIR, exist_ok=True)
TEST_ORIGINS = test_o                     # fixed here so every later cell scores the same set
SEEDS_N = 1 if SMOKE else SCFG['seeds']
EPOCHS  = 3 if SMOKE else SCFG['epochs']
print('output   ->', PROTO_DIR)
print('test origins:', len(TEST_ORIGINS))

GRAPH_MODES = ('none', 'phys', 'static', 'context')

class CMILModes(CMIL):
    """CMIL with the adjacency source switchable.

      none    : no message passing at all (encoder-only rung)
      phys    : A_t = A_phys, fixed for all t
      static  : A_t = softmax(W), learned but context-independent and time-invariant
      context : unchanged v4 behaviour

    A price-conditioned graph is NOT a mode here. It is obtained by feeding price-only
    context (ctx='price') into the unchanged context mode, which is the same thing with
    fewer moving parts and no masking of an already-pooled context vector.
    """
    def __init__(self, *a, graph_mode='context', **kw):
        super().__init__(*a, **kw)
        assert graph_mode in GRAPH_MODES, graph_mode
        self.graph_mode = graph_mode
        if graph_mode == 'static':
            self.static_logits = nn.Parameter(torch.randn(self.R, self.R) * 0.1)

    def forward(self, H, Sbar, C, Aphys, xg):
        if self.graph_mode == 'context':
            return super().forward(H, Sbar, C, Aphys, xg)

        B = H.shape[0]
        G = torch.einsum('nr,bnd->brd', Sbar, H) + self.city_emb.unsqueeze(0)
        if self.graph_mode == 'none':
            # Return the city representations untouched; the fusion gate in CMINScenario
            # then decides how much of a no-op neighbourhood to use.
            eye = torch.eye(self.R, device=H.device).unsqueeze(0).expand(B, -1, -1)
            return G, eye, {'Actx': eye, 'alpha': torch.ones(B, 1, 1, device=H.device),
                            'temperature': torch.ones(B, 1, 1, device=H.device),
                            'head_w': torch.zeros(B, self.n_heads, device=H.device)}

        Cg = C
        if self.graph_mode == 'phys':
            At = Aphys.unsqueeze(0).expand(B, -1, -1)
            V = self.Wv(G)
            aux = {'Actx': At, 'alpha': torch.ones(B, 1, 1, device=H.device),
                   'temperature': torch.ones(B, 1, 1, device=H.device),
                   'head_w': torch.zeros(B, self.n_heads, device=H.device)}
            return torch.matmul(At, V), At, aux

        # only 'static' reaches here
        Actx = F.softmax(self.static_logits, dim=-1).unsqueeze(0).expand(B, -1, -1)

        raw = (torch.sigmoid(self.alpha_head(Cg)).view(-1, 1, 1) if self.context_alpha
               else torch.sigmoid(self.alpha_logit))
        alpha = self.alpha_min + self.alpha_span * raw
        At = alpha * Aphys.unsqueeze(0) + (1.0 - alpha) * Actx
        V = self.Wv(G)
        return torch.matmul(At, V), At, {'Actx': Actx, 'alpha': alpha,
                                         'temperature': torch.ones(B, 1, 1, device=H.device),
                                         'head_w': torch.zeros(B, self.n_heads, device=H.device)}


# The nine rungs. `ctx` selects which context reaches the model at all; `graph` selects what
# shapes the adjacency. Rungs 5 and 6 differ ONLY in `graph`, which is the decisive contrast.
LADDER = {
    'L1_encoder_only':      dict(ctx='price', graph='none',    anchor=True),
    'L2_fixed_physical':    dict(ctx='price', graph='phys',    anchor=True),
    'L3_learned_static':    dict(ctx='price', graph='static',  anchor=True),
    'L4_price_dynamic':     dict(ctx='price', graph='context', anchor=True),
    'L5_ctx_features_only': dict(ctx='full',  graph='phys',    anchor=True),   # <-- control
    'L6_ctx_conditioned':   dict(ctx='full',  graph='context', anchor=True),   # <-- claim
    'L7_ctx_shuffled':      dict(ctx='full',  graph='context', anchor=True, placebo='time'),
    'L8_no_phys_anchor':    dict(ctx='full',  graph='context', anchor=False),
    # L9 (market-level vs city-level graph) is deliberately omitted. The protocol admits it
    # "only if both can be implemented without changing other components", and a market-level
    # graph replaces the pooling and unpooling stages -- so it would not be the same model
    # with a different graph. Report it as not run rather than as a like-for-like rung.
}
# Every rung is run under BOTH forecast heads. v4's Table 9 established that the residual
# parameterisation owns h <= 14 and the standard one owns h >= 14, so a ladder fixed to one
# head would be judged in the regime where that head is known to be weak -- which is exactly
# how the smoke run ended up showing a plain STGCN beating every rung at h=30.
HEADS = {'R': dict(residual_head=True),      # persistence-anchored: strong at short h
         'S': dict(residual_head=False)}     # standard decoder: strong at long h

DECISIVE = ('L5_ctx_features_only', 'L6_ctx_conditioned')

def rung_name(base, head):
    return '%s@%s' % (base, head)

print('ladder rungs:', len(LADDER))
print('decisive contrast: %s  vs  %s' % DECISIVE)
print('SMOKE =', SMOKE, '| seeds =', 1 if SMOKE else SCFG['seeds'],
      '| epochs =', 3 if SMOKE else SCFG['epochs'])

output   -> /content/drive/MyDrive/cmin_v4_out/protocol
test origins: 337
ladder rungs: 8
decisive contrast: L5_ctx_features_only  vs  L6_ctx_conditioned
SMOKE = True | seeds = 1 | epochs = 3


In [15]:
# === P2. Binding the ladder to the v4 machinery ===
# Every rung shares one temporal encoder, one context encoder, one fusion gate and one head.
# What differs is (a) which context reaches the model and (b) what shapes the adjacency.
# Nothing else may vary, or the comparison stops isolating the mechanism.
LADDER_SC = dict(residual_head=True, horizon_weight=True)

def ladder_arrays(cfg):
    """Arrays for one rung.

    ctx='price' zeroes the non-price node blocks and the whole global block instead of
    deleting them. Deleting would change Fn/Fg and therefore parameter count, which would
    confound 'less context' with 'smaller model' -- the exact confound this ladder exists
    to remove.
    """
    xn, xg, ns, gs, dom = scenario_arrays(dict(LADDER_SC))
    if cfg['ctx'] == 'price':
        xn = xn.copy(); xg = np.zeros_like(xg)
        for k, idx in ns.items():
            if k != 'price':
                xn[..., list(idx)] = 0.0
    if cfg.get('placebo') == 'time':
        # v4's apply_placebo only routes ('news','policy','polnews','news_node'). Asking it
        # for 'economy' was a silent no-op, so L7 came out BYTE-IDENTICAL to L6 --
        # 2185/4867/7886/12184/19319 at every horizon. Shift the context here instead.
        # A circular shift, not a permutation: it preserves each channel's autocorrelation
        # and marginals exactly and destroys only the alignment to price, which is the one
        # thing this control is meant to remove.
        off = int(0.37 * xg.shape[0])
        xn = xn.copy()
        xg = np.roll(xg, off, axis=0)
        for k, idx in ns.items():
            if k != 'price':
                cols = list(idx)
                xn[..., cols] = np.roll(xn[..., cols], off, axis=0)
    return xn, xg, ns, gs, dom

def build_ladder_model(cfg, head='R'):
    xn, xg, ns, gs, dom = ladder_arrays(cfg)
    sc = dict(LADDER_SC); sc.update(HEADS[head])
    m = CMINScenario(xn.shape[-1], xg.shape[-1], CFG['d'], CFG['d_ctx'], CFG['H'],
                     ns, gs, dom, sc)
    m.cmil = CMILModes(CFG['d'], CFG['d_ctx'], R, xg.shape[-1],
                       n_heads=CFG.get('n_heads', 4),
                       context_alpha=CFG.get('context_alpha', True),
                       graph_mode=cfg['graph'])
    if not cfg.get('anchor', True):
        # alpha pinned to 0 -> the physical graph contributes nothing and A_t is purely learned
        m.cmil.alpha_min, m.cmil.alpha_span = 0.0, 0.0
    return m.to(DEVICE), (xn, xg)

def rung_ckpt(name, seed):
    return os.path.join(PROTO_DIR, 'w_%s__s%d.pt' % (name, seed))

def train_ladder_rung(cfg, seed, epochs=None, name=None, head='R'):
    torch.manual_seed(seed); np.random.seed(seed)
    epochs = epochs or SCFG['epochs']
    model, (xn, xg) = build_ladder_model(cfg, head)
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'],
                           weight_decay=CFG['weight_decay'])
    dtr = make_loader_s(train_o, True,  xn, xg)
    dva = make_loader_s(val_o,   False, xn, xg)
    dte = make_loader_s(TEST_ORIGINS, False, xn, xg)

    def fwd(b):
        xnb, xgb, y, ob, spk, anc = [z.to(DEVICE) for z in b]
        out = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
        p = out[0] if isinstance(out, (tuple, list)) else out
        return p, y, ob

    def score(dl):
        model.eval(); num = den = 0.0
        with torch.no_grad():
            for b in dl:
                p, y, ob = fwd(b)
                w = HW_t.view(1, 1, -1)
                num += ((p - y).abs() * ob * w).sum().item()
                den += (ob * w).sum().item()
        return num / max(den, 1e-9)

    best, bad, bstate = np.inf, 0, None
    for ep in range(epochs):
        model.train()
        for b in dtr:
            p, y, ob = fwd(b)
            loss = ((p - y).abs() * ob * HW_t.view(1, 1, -1)).sum() / (
                (ob * HW_t.view(1, 1, -1)).sum().clamp(min=1e-9))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()
        v = score(dva)
        if v < best - 1e-6:
            best, bad = v, 0
            bstate = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= (SCFG['patience'] if epochs > 5 else 2):
                break
    if bstate:
        model.load_state_dict(bstate)
        if name:
            torch.save(bstate, rung_ckpt(name, seed))

    preds = []
    model.eval()
    with torch.no_grad():
        for b in dte:
            p, _, _ = fwd(b)
            preds.append(p.cpu().numpy())
    P = np.concatenate(preds, 0)
    assert P.shape == (len(TEST_ORIGINS), Y.shape[1], CFG['H']), P.shape
    return P

def load_rung_weights(model, name, seed):
    f = rung_ckpt(name, seed)
    if os.path.exists(f):
        model.load_state_dict(torch.load(f, map_location=DEVICE)); return True
    return False

print('ladder glue ready: ladder_arrays / build_ladder_model / train_ladder_rung')

ladder glue ready: ladder_arrays / build_ladder_model / train_ladder_rung


In [16]:
# === P3. Baseline families (protocol Table 4) ===
# Registered independently so a failure in one records a reason and the suite continues.
# The statistical ones are exact; the neural ones are compact re-implementations matched on
# encoder width, optimiser, budget and selection rule -- not the authors' released code.
import traceback

BASELINES = {}
def baseline(name, family):
    def deco(fn):
        BASELINES[name] = dict(fn=fn, family=family)
        return fn
    return deco

def _yhat_shape(oo):
    return (len(oo), Y.shape[1], CFG['H'])

# ---------------------------------------------------------------- naive / statistical ---
@baseline('Persistence', 'naive')
def _persist(tr, te):
    return np.stack([np.repeat(Y[t][:, None], CFG['H'], 1) for t in te])

@baseline('SeasonalNaive7', 'naive')
def _snaive(tr, te):
    out = np.zeros(_yhat_shape(te))
    for k, t in enumerate(te):
        for h in range(1, CFG['H'] + 1):
            out[k, :, h - 1] = Y[t - 7 + ((h - 1) % 7) + 1]
    return out

@baseline('Drift', 'naive')
def _drift(tr, te):
    out = np.zeros(_yhat_shape(te))
    for k, t in enumerate(te):
        slope = (Y[t] - Y[t - 30]) / 30.0
        for h in range(1, CFG['H'] + 1):
            out[k, :, h - 1] = Y[t] + slope * h
    return out

@baseline('ETS', 'statistical')
def _ets(tr, te):
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    end = min(te)
    out = np.zeros(_yhat_shape(te))
    fc = np.zeros((Y.shape[1], CFG['H']))
    for n in range(Y.shape[1]):
        s = pd.Series(Y[:end, n]).ffill().bfill()
        try:
            f = ExponentialSmoothing(s, trend='add', damped_trend=True,
                                     seasonal=None).fit(optimized=True)
            fc[n] = f.forecast(CFG['H']).to_numpy()
        except Exception:
            fc[n] = s.iloc[-1]
    # ETS is refit once on the training portion; re-fitting per origin over 120 markets
    # and hundreds of origins is not affordable and would not change the ranking.
    for k, t in enumerate(te):
        out[k] = Y[t][:, None] + (fc - fc[:, :1])
    return out

# ------------------------------------------------------------------- machine learning ---
@baseline('XGBoost', 'ml')
def _xgb(tr, te):
    from xgboost import XGBRegressor
    lags = [1, 2, 3, 5, 7, 14, 30]
    def feats(origins):
        F_ = [np.stack([Y[np.array(origins) - L] for L in lags], -1)]      # (T,N,lags)
        g = Xglobal[np.array(origins)]                                     # (T,Fg)
        F_.append(np.repeat(g[:, None, :], Y.shape[1], 1))
        return np.concatenate(F_, -1)
    Xtr, Xte = feats(tr), feats(te)
    out = np.zeros(_yhat_shape(te))
    for h in range(1, CFG['H'] + 1):
        ytr = np.stack([Y[np.array(tr) + h]])[0]
        m = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, n_jobs=4,
                         tree_method='hist', verbosity=0)
        m.fit(Xtr.reshape(-1, Xtr.shape[-1]), ytr.reshape(-1))
        out[:, :, h - 1] = m.predict(Xte.reshape(-1, Xte.shape[-1])).reshape(len(te), -1)
    return out

# --------------------------------------------------------------------- temporal neural --
class _Seq(nn.Module):
    def __init__(self, kind, Fn, d, H):
        super().__init__()
        self.kind = kind
        if kind in ('lstm', 'gru'):
            self.rnn = (nn.LSTM if kind == 'lstm' else nn.GRU)(Fn, d, batch_first=True)
        else:                                            # tcn
            self.rnn = DilatedTemporalEncoder(Fn, d)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))
    def forward(self, xn, xg=None, *a):
        B, N, L_, Fn = xn.shape
        x = xn.reshape(B * N, L_, Fn)
        z = self.rnn(x) if self.kind == 'tcn' else self.rnn(x)[0][:, -1]
        return self.head(z).reshape(B, N, -1)

class _NHiTS(nn.Module):
    """Compact N-HiTS: pooled multi-rate MLP stacks with residual subtraction."""
    def __init__(self, Fn, d, H, rates=(1, 2, 4)):
        super().__init__()
        self.rates = rates
        self.blocks = nn.ModuleList()
        for r in rates:
            self.blocks.append(nn.Sequential(
                nn.Linear(max(1, CFG['L'] // r), d), nn.ReLU(),
                nn.Linear(d, d), nn.ReLU(), nn.Linear(d, CFG['L'] + H)))
    def forward(self, xn, xg=None, *a):
        B, N, L_, _ = xn.shape
        res = xn[..., 0].reshape(B * N, L_)
        out = 0.0
        for r, blk in zip(self.rates, self.blocks):
            p = F.avg_pool1d(res.unsqueeze(1), r).squeeze(1) if r > 1 else res
            z = blk(p)
            back, fore = z[:, :L_], z[:, L_:]
            res = res - back
            out = out + fore
        return out.reshape(B, N, -1)

# ------------------------------------------------------------------------ graph neural --
class _STGCN(nn.Module):
    """Fixed-graph STGNN: temporal encoder + propagation over A_phys only."""
    def __init__(self, Fn, d, H, A):
        super().__init__()
        self.enc = DilatedTemporalEncoder(Fn, d)
        self.register_buffer('A', A)
        self.gc = nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))
    def forward(self, xn, xg=None, *a):
        B, N, L_, Fn = xn.shape
        h = self.enc(xn.reshape(B * N, L_, Fn)).reshape(B, N, -1)
        h = h + torch.relu(self.gc(torch.einsum('ij,bjd->bid', self.A, h)))
        return self.head(h)

class _AdaptiveGraph(nn.Module):
    """Adaptive-graph family (AGCRN / Graph WaveNet): adjacency from learned node
    embeddings, context-independent by construction. `mode` only changes the temporal
    backbone -- gru for AGCRN-style, tcn for Graph WaveNet-style."""
    def __init__(self, Fn, d, H, N, mode='tcn', emb=16):
        super().__init__()
        self.mode = mode
        self.E = nn.Parameter(torch.randn(N, emb) * 0.1)
        self.enc = (DilatedTemporalEncoder(Fn, d) if mode == 'tcn'
                    else nn.GRU(Fn, d, batch_first=True))
        self.gc = nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))
    def forward(self, xn, xg=None, *a):
        B, N, L_, Fn = xn.shape
        x = xn.reshape(B * N, L_, Fn)
        h = (self.enc(x) if self.mode == 'tcn' else self.enc(x)[0][:, -1]).reshape(B, N, -1)
        A = F.softmax(torch.relu(self.E @ self.E.t()), dim=-1)
        h = h + torch.relu(self.gc(torch.einsum('ij,bjd->bid', A, h)))
        return self.head(h)

print('registered %d statistical/ML baselines' % len(BASELINES))
print('neural baselines: LSTM, GRU, TCN, N-HiTS, STGCN(A_phys), AGCRN-style, GraphWaveNet-style')

registered 5 statistical/ML baselines
neural baselines: LSTM, GRU, TCN, N-HiTS, STGCN(A_phys), AGCRN-style, GraphWaveNet-style


In [17]:
# === P4. Register the neural and graph baselines ===
# The previous cell only DEFINES the model classes. Without this cell they are never
# trained, and Table 4's temporal-neural, fixed-graph and adaptive-graph rows stay empty --
# which is precisely the gap the protocol was written to close. The smoke run caught it:
# five naive/ML rows, and nothing from any neural family.
BASE_XN, BASE_XG, _ns, _gs, _dom = ladder_arrays(dict(ctx='full', graph='phys'))

# Market-level physical adjacency, pushed down from the city graph through the assignment
# matrix, because these baselines propagate over markets rather than over cities.
A_MKT = S @ A_phys @ S.T
np.fill_diagonal(A_MKT, 0.0)
A_MKT = A_MKT / np.maximum(A_MKT.sum(1, keepdims=True), 1e-8)
A_MKT_t = torch.tensor(A_MKT, dtype=torch.float32, device=DEVICE)
print('market-level A_phys:', A_MKT.shape, 'density %.3f' % (A_MKT > 0).mean())

def train_torch_baseline(factory, seed, epochs=None):
    """Same loaders, loss, horizon weights, optimiser and selection rule as the ladder, so
    a baseline never loses on budget rather than on merit."""
    torch.manual_seed(seed); np.random.seed(seed)
    epochs = epochs or SCFG['epochs']
    model = factory().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    dtr = make_loader_s(train_o, True,  BASE_XN, BASE_XG)
    dva = make_loader_s(val_o,   False, BASE_XN, BASE_XG)
    dte = make_loader_s(TEST_ORIGINS, False, BASE_XN, BASE_XG)
    W = HW_t.view(1, 1, -1)

    def step(b):
        xnb, xgb, y, ob, spk, anc = [z.to(DEVICE) for z in b]
        p = model(xnb, xgb) + anc.unsqueeze(-1)      # same persistence anchor as the ladder
        return p, y, ob

    def val():
        model.eval(); n = d = 0.0
        with torch.no_grad():
            for b in dva:
                p, y, ob = step(b)
                n += ((p - y).abs() * ob * W).sum().item(); d += (ob * W).sum().item()
        return n / max(d, 1e-9)

    best, bad, bstate = np.inf, 0, None
    for ep in range(epochs):
        model.train()
        for b in dtr:
            p, y, ob = step(b)
            loss = ((p - y).abs() * ob * W).sum() / (ob * W).sum().clamp(min=1e-9)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()
        v = val()
        if v < best - 1e-6:
            best, bad = v, 0
            bstate = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= (SCFG['patience'] if epochs > 5 else 2): break
    if bstate: model.load_state_dict(bstate)
    model.eval(); out = []
    with torch.no_grad():
        for b in dte:
            p, _, _ = step(b)
            out.append(p.cpu().numpy())
    P = np.concatenate(out, 0)
    assert P.shape == (len(TEST_ORIGINS), Y.shape[1], CFG['H']), P.shape
    return P

Fn_b, d_, H_ = BASE_XN.shape[-1], CFG['d'], CFG['H']
NEURAL_FAMILIES = {
    'LSTM':        ('temporal',  lambda: _Seq('lstm', Fn_b, d_, H_)),
    'GRU':         ('temporal',  lambda: _Seq('gru',  Fn_b, d_, H_)),
    'TCN':         ('temporal',  lambda: _Seq('tcn',  Fn_b, d_, H_)),
    'NHiTS':       ('temporal',  lambda: _NHiTS(Fn_b, d_, H_)),
    'STGCN_phys':  ('fixed',     lambda: _STGCN(Fn_b, d_, H_, A_MKT_t)),
    'AGCRN_style': ('adaptive',  lambda: _AdaptiveGraph(Fn_b, d_, H_, Y.shape[1], mode='gru')),
    'GWNet_style': ('adaptive',  lambda: _AdaptiveGraph(Fn_b, d_, H_, Y.shape[1], mode='tcn')),
}
for _n, (_fam, _f) in NEURAL_FAMILIES.items():
    # no 'fn': the runner dispatches on the presence of 'factory'
    BASELINES[_n] = dict(family=_fam, factory=_f)

print('registered %d baselines total' % len(BASELINES))
for _f in ('naive', 'statistical', 'ml', 'temporal', 'fixed', 'adaptive'):
    print('   %-12s %s' % (_f, [k for k, v in BASELINES.items() if v['family'] == _f]))
print('')
print('Dynamic-graph family: L4_price_dynamic in the ladder is the comparator,')
print('protocol allows ("or a feasible contemporary comparator").')

market-level A_phys: (120, 120) density 0.203
registered 12 baselines total
   naive        ['Persistence', 'SeasonalNaive7', 'Drift']
   statistical  ['ETS']
   ml           ['XGBoost']
   temporal     ['LSTM', 'GRU', 'TCN', 'NHiTS']
   fixed        ['STGCN_phys']
   adaptive     ['AGCRN_style', 'GWNet_style']

Dynamic-graph family: L4_price_dynamic in the ladder is the comparator,
protocol allows ("or a feasible contemporary comparator").


In [18]:
# === P5. Metric suite (protocol 5.5) ===
# MAE and RMSE in IDR, MASE and skill against persistence, and the distributional views the
# protocol asks for: per-market spread, the worst market decile, high-volatility windows,
# and price-spike days. Aggregate MAE alone hides exactly the cases a food-price model is
# bought for.
# The protocol reports MAE in IDR. Y is per-market z-scored log price, so a metric computed
# on it is in z-units and is NOT comparable to v4's Table 6 -- the smoke run made this
# obvious by printing a whole table of 0.0 and 1.0. This is the numpy twin of v4's
# to_rp(z) = exp(z*sd + mu), applied to predictions and targets before any error is taken.
def RP(z):
    return np.exp(z * sd[None, :, None] + mu[None, :, None])

def metric_block(pred, te, tag=''):
    """pred: (T_te, N, H) in model units. Returns a dict of every protocol metric."""
    yt = RP(np.stack([Y[t + 1:t + 1 + CFG['H']].T for t in te]))    # (T,N,H) in IDR
    ob = np.stack([obs[t + 1:t + 1 + CFG['H']].T for t in te])
    per = RP(np.stack([np.repeat(Y[t][:, None], CFG['H'], 1) for t in te]))
    pred = RP(pred)
    out = {}
    for hi, h in enumerate(CFG['HORIZONS']):
        k = h - 1
        e  = np.abs(pred[:, :, k] - yt[:, :, k]) * ob[:, :, k]
        ep = np.abs(per[:, :, k]  - yt[:, :, k]) * ob[:, :, k]
        n  = np.maximum(ob[:, :, k].sum(), 1)
        mae  = e.sum() / n
        rmse = np.sqrt((((pred[:, :, k] - yt[:, :, k]) ** 2) * ob[:, :, k]).sum() / n)
        mase = mae / max(ep.sum() / n, 1e-12)
        # per-market MAE, for the spread and worst-decile views
        pm = e.sum(0) / np.maximum(ob[:, :, k].sum(0), 1)
        # high-volatility windows: top quintile of realised |dlog| over the target span
        vol = np.abs(np.diff(yt[:, :, :k + 1], axis=2)).mean((1, 2)) if k > 0 else np.zeros(len(te))
        hv = vol >= np.quantile(vol, 0.8) if k > 0 else np.zeros(len(te), bool)
        # spike defined on the log scale, as in v4, not on the Rupiah difference
        spike = np.abs(np.log(np.maximum(yt[:, :, k], 1e-9))
                       - np.log(np.maximum(per[:, :, k], 1e-9))) >= SCFG['spike_threshold']
        out[h] = dict(
            MAE=float(mae), RMSE=float(rmse), MASE=float(mase),
            skill_vs_persistence=float(1 - mae / max(ep.sum() / n, 1e-12)),
            market_MAE_sd=float(pm.std()),
            worst_decile_MAE=float(np.quantile(pm, 0.9)),
            highvol_MAE=float((e[hv].sum() / max(ob[:, :, k][hv].sum(), 1)) if hv.any() else np.nan),
            spike_MAE=float((e * spike).sum() / max((ob[:, :, k] * spike).sum(), 1)),
            spike_share=float(spike.mean()))
    return out

def metrics_frame(store):
    rows = []
    for (name, seed), pred in store.items():
        for h, m in metric_block(pred, TEST_ORIGINS).items():
            rows.append(dict(model=name, seed=seed, h=h, **m))
    return pd.DataFrame(rows)

print('metrics: MAE, RMSE, MASE, skill, market spread, worst decile, high-volatility, spikes')

metrics: MAE, RMSE, MASE, skill, market spread, worst decile, high-volatility, spikes


In [19]:
# === P6. Run everything, resumable ===
import pickle, time, traceback

def pth(name, seed):
    return os.path.join(PROTO_DIR, '%s__s%d.pkl' % (name.replace('+', '_'), seed))

STORE, FAILED = {}, {}
def record(name, seed, pred, meta=None):
    with open(pth(name, seed), 'wb') as fh:
        pickle.dump(dict(pred=pred.astype(np.float32), meta=meta or {}), fh)
    STORE[(name, seed)] = pred

def load_done():
    for f in os.listdir(PROTO_DIR):
        if not f.endswith('.pkl'): continue
        name, s = f[:-4].rsplit('__s', 1)
        with open(os.path.join(PROTO_DIR, f), 'rb') as fh:
            STORE[(name, int(s))] = pickle.load(fh)['pred']
load_done()
print('%d runs already on disk' % len(STORE))

t0 = time.time()
for name, spec in BASELINES.items():
    for seed in range(1 if spec['family'] in ('naive', 'statistical') else SEEDS_N):
        if (name, seed) in STORE: continue
        try:
            pred = (train_torch_baseline(spec['factory'], seed, epochs=EPOCHS)
                    if 'factory' in spec else spec['fn'](train_o, TEST_ORIGINS))
            record(name, seed, pred, dict(family=spec['family']))
            print('  %-18s s%d  ok  (%.0fs)' % (name, seed, time.time() - t0))
        except Exception as e:
            FAILED[name] = '%s: %s' % (type(e).__name__, e)
            print('  %-18s SKIPPED -> %s' % (name, FAILED[name]))
            break

# A control that silently equals the thing it controls is worse than no control, so verify
# the arrays really differ before spending GPU time on the rung.
_a = ladder_arrays(LADDER['L6_ctx_conditioned'])[1]
_b = ladder_arrays(LADDER['L7_ctx_shuffled'])[1]
assert not np.allclose(_a, _b), 'L7 placebo inert: its context equals L6'
print('placebo check: L7 context differs from L6, max |diff| %.3f' % np.abs(_a - _b).max())

print('\n=== ladder ===')
for head in HEADS:
    for base, cfg in LADDER.items():
        name = rung_name(base, head)
        for seed in range(SEEDS_N):
            if (name, seed) in STORE: continue
            try:
                pred = train_ladder_rung(cfg, seed, epochs=EPOCHS, name=name, head=head)
                record(name, seed, pred, dict(cfg, head=head))
                print('  %-26s s%d  ok  (%.0fs)' % (name, seed, time.time() - t0))
            except Exception as e:
                FAILED[name] = '%s: %s' % (type(e).__name__, e)
                print('  %-26s FAILED -> %s' % (name, FAILED[name]))
                traceback.print_exc(limit=2)
                break

print('\ndone in %.1f min | %d runs stored | %d models skipped'
      % ((time.time() - t0) / 60, len(STORE), len(FAILED)))
for k, v in FAILED.items():
    print('  SKIPPED %-22s %s' % (k, v))

0 runs already on disk
  Persistence        s0  ok  (0s)
  SeasonalNaive7     s0  ok  (0s)
  Drift              s0  ok  (0s)
  ETS                s0  ok  (14s)
  XGBoost            s0  ok  (198s)
  LSTM               s0  ok  (206s)
  GRU                s0  ok  (208s)
  TCN                s0  ok  (219s)
  NHiTS              s0  ok  (221s)
  STGCN_phys         s0  ok  (231s)
  AGCRN_style        s0  ok  (233s)
  GWNet_style        s0  ok  (244s)
placebo check: L7 context differs from L6, max |diff| 3.890

=== ladder ===
  L1_encoder_only@R          s0  ok  (256s)
  L2_fixed_physical@R        s0  ok  (267s)
  L3_learned_static@R        s0  ok  (279s)
  L4_price_dynamic@R         s0  ok  (291s)
  L5_ctx_features_only@R     s0  ok  (303s)
  L6_ctx_conditioned@R       s0  ok  (315s)
  L7_ctx_shuffled@R          s0  ok  (328s)
  L8_no_phys_anchor@R        s0  ok  (340s)
  L1_encoder_only@S          s0  ok  (351s)
  L2_fixed_physical@S        s0  ok  (363s)
  L3_learned_static@S        s0  ok 

In [20]:
# === P7. Tables 4 and 5, and the decisive contrast ===
M = metrics_frame(STORE)
M.to_csv(os.path.join(PROTO_DIR, 'metrics_long.csv'), index=False)

print('=== Table 5: multi-horizon MAE (IDR), mean over seeds, sd in brackets ===')
t5 = M.pivot_table(index='model', columns='h', values='MAE', aggfunc=['mean', 'std'])
print(t5.round(0).to_string())

print('\n=== Table 4 status ===')
fam = {n: BASELINES[n]['family'] for n in BASELINES}
for n in BASELINES:
    st = 'done (%d seeds)' % len({s for (m, s) in STORE if m == n}) if any(
        m == n for (m, s) in STORE) else 'SKIPPED: %s' % FAILED.get(n, 'not run')
    print('  %-12s %-18s %s' % (fam[n], n, st))

print('\n=== THE DECISIVE CONTRAST (protocol 5.3, rungs 5 vs 6) ===')
a, b = DECISIVE
for head in HEADS:
    ra, rb = rung_name(a, head), rung_name(b, head)
    sub = M[M.model.isin([ra, rb])]
    label = 'persistence-anchored' if head == 'R' else 'standard decoder'
    print('')
    print('--- forecast head %s (%s) ---' % (head, label))
    if sub.model.nunique() < 2:
        print('    both rungs not yet run')
        continue
    piv = sub.pivot_table(index='h', columns='model', values='MAE', aggfunc='mean')
    piv['delta_pct'] = 100 * (1 - piv[rb] / piv[ra])
    print(piv.round(2).to_string())
    dd = piv['delta_pct']
    if (dd.abs() < 1.0).all():
        print('    [!] every horizon within 1 pct: graph conditioning is NOT carrying the')
        print('        result. Context is behaving as ordinary extra features here.')
    else:
        print('    conditioning changes accuracy by %.1f to %.1f pct across horizons.'
              % (dd.min(), dd.max()))
print('')
print('delta_pct > 0 means context-CONDITIONING beats context-as-features.')
print('Read each head only in the range v4 Table 9 gives it: R for h<=14, S for h>=14.')

# LaTeX for direct paste into the manuscript
tex = t5['mean'].round(0).to_latex(caption='Multi-horizon MAE in IDR.', label='tab:main')
open(os.path.join(PROTO_DIR, 'table5.tex'), 'w', encoding='utf-8').write(tex)
print('\nwrote metrics_long.csv and table5.tex ->', PROTO_DIR)

=== Table 5: multi-horizon MAE (IDR), mean over seeds, sd in brackets ===
                          mean                                   
h                           1       3        7        14       30
model                                                            
AGCRN_style             2223.0  4868.0   7810.0  11234.0  14455.0
Drift                   2499.0  5640.0  10184.0  18834.0  43348.0
ETS                     2036.0  4751.0   8139.0  13110.0  22175.0
GRU                     2148.0  4808.0   7828.0  11744.0  14947.0
GWNet_style             2386.0  4827.0   8094.0  12185.0  15478.0
L1_encoder_only@R       2147.0  4764.0   7714.0  11551.0  17074.0
L1_encoder_only@S       2744.0  5060.0   7507.0  11146.0  16374.0
L2_fixed_physical@R     2110.0  4791.0   7941.0  11866.0  18156.0
L2_fixed_physical@S     3105.0  4992.0   7539.0  11519.0  16634.0
L3_learned_static@R     2270.0  4782.0   8004.0  11827.0  18196.0
L3_learned_static@S     4695.0  5773.0   7687.0  11447.0  16567.0
L4

In [21]:
# === P8. Graph validation (protocol 7) ===
# 7.1 stability across seeds, 7.2 alignment with external structure, 7.3 event-aligned
# change, 7.4 null controls. Reported as diagnostics, never as causal transport: a stable
# learned adjacency is predictive information flow under the model and nothing more.
def graph_snapshots(name, seed, n_origin=64):
    base, head = name.split('@')
    cfg = LADDER[base]
    m, (xn, xg) = build_ladder_model(cfg, head)
    if not load_rung_weights(m, name, seed):
        print('  no saved weights for %s s%d' % (name, seed)); return None
    m.eval()
    As = []
    with torch.no_grad():
        for b in make_loader_s(TEST_ORIGINS[:n_origin], False, xn, xg):
            xnb, xgb = b[0].to(DEVICE), b[1].to(DEVICE)
            out = m(xnb, xgb, Sbar_t, S_t, Aphys_t)
            As.append(out[1].cpu().numpy())
    return np.concatenate(As, 0)

def validate_graph(cfg_name):
    cfg = LADDER[cfg_name.split('@')[0]]
    seeds = sorted({s for (m, s) in STORE if m == cfg_name})
    if len(seeds) < 2:
        print('need >=2 seeds for stability'); return
    snaps = {s: graph_snapshots(cfg_name, s) for s in seeds}
    snaps = {k: v.mean(0) for k, v in snaps.items() if v is not None}
    seeds = sorted(snaps)
    if len(seeds) < 2:
        print('need >=2 seeds with saved weights'); return
    print('=== 7.1 stability across seeds ===')
    cors = [np.corrcoef(snaps[a].ravel(), snaps[b].ravel())[0, 1]
            for i, a in enumerate(seeds) for b in seeds[i + 1:]]
    print('  mean pairwise edge correlation: %.3f over %d pairs' % (np.mean(cors), len(cors)))

    A0 = np.mean(list(snaps.values()), 0)
    print('\n=== 7.2 alignment with the physical graph ===')
    ph = A_phys
    print('  corr(learned, A_phys)          : %.3f' % np.corrcoef(A0.ravel(), ph.ravel())[0, 1])
    print('  mass on physical top-8 edges   : %.3f'
          % A0[ph > np.quantile(ph, 1 - 8.0 / ph.shape[0])].sum() / A0.sum())

    print('\n=== 7.4 null controls ===')
    print('  A random row-stochastic graph and a shuffled-context run are the two nulls;')
    print('  rungs L7_ctx_shuffled and a uniform-A variant supply them. Compare their')
    print('  MAE against L6 before treating any learned edge as meaningful.')
    return A0

print('graph validation ready: validate_graph("L6_ctx_conditioned@S") after the run')

graph validation ready: validate_graph("L6_ctx_conditioned@S") after the run


## Order of work

1. `SMOKE = True`, run every cell top to bottom. Expect ~10 minutes. Fix anything in the
   skip list before going further — a model that fails here fails identically four hours in.
2. `SMOKE = False`, re-run cells 14–16. Checkpoints make this resumable; a disconnect costs
   one run.
3. Read the decisive contrast in cell 15 **before** writing any of Section 6.

## If rungs 5 and 6 come out equal

That is a real possible outcome and it is worth planning for now rather than discovering it
during the write-up. It would mean the numerical context helps as ordinary features and not
by reshaping the graph — so the honest paper is *"context-conditioned graph learning does not
outperform context-as-features on this panel, and here is the controlled evidence"*, which is
publishable and considerably harder to referee away than an unaudited claim of the opposite.

The existing v4 finding supports taking this seriously: the global context gate puts 0.9888
of its mass on the price component, so the context encoder is already known to be degenerate
under the standard head. A mechanism that saturates is unlikely to be the thing driving the
gains.

## What still has to be written by hand

This notebook produces numbers, not prose. Sections 2 (related work), 3 (data), 4 (method)
carry over from the current draft with the protocol's Tables 1 and 2 added. Section 7.5, on
the limits of graph interpretation, should stay close to the protocol's wording — it is the
most defensible paragraph in that document.